# Setup and configuration

In [2]:
import numpy as np
import pandas as pd

from pathlib import Path
from scipy import stats

In [3]:
ROOT_DIR = Path.cwd()

STAGE1_OUTPUT_DIR = (
    ROOT_DIR
    / "outputs"
    / "stage1"
)

STAGE2_OUTPUT_DIR = (
    ROOT_DIR
    / "outputs"
    / "stage2"
)

STAGE2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [4]:
MAXIMUM_IDEA_GAP_MINUTES = 3.0
TRADING_INTENSITY_WINDOW_MINUTES = 60

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [5]:
if not STAGE1_OUTPUT_DIR.exists():
    raise FileNotFoundError(
        "Stage 1 output directory was not found: "
        f"{STAGE1_OUTPUT_DIR}"
    )

print(
    "Stage 1 input directory:",
    STAGE1_OUTPUT_DIR,
)

print(
    "Stage 2 output directory:",
    STAGE2_OUTPUT_DIR,
)

Stage 1 input directory: /Users/charltonsiaw/Desktop/C22-veNTUre/outputs/stage1
Stage 2 output directory: /Users/charltonsiaw/Desktop/C22-veNTUre/outputs/stage2


# Load stage 1 outputs

In [6]:
stage1_file_paths = {
    "trades": (
        STAGE1_OUTPUT_DIR
        / "trades.parquet"
    ),
    "trades_with_ideas": (
        STAGE1_OUTPUT_DIR
        / "trades_with_ideas.parquet"
    ),
    "trades_with_previous_completed": (
        STAGE1_OUTPUT_DIR
        / "trades_with_previous_completed.parquet"
    ),
    "idea_features": (
        STAGE1_OUTPUT_DIR
        / "idea_features_stage1.parquet"
    ),
}

In [7]:
trades = pd.read_parquet(
    stage1_file_paths["trades"]
)

trades_with_ideas = pd.read_parquet(
    stage1_file_paths[
        "trades_with_ideas"
    ]
)

trades_with_previous_completed = (
    pd.read_parquet(
        stage1_file_paths[
            "trades_with_previous_completed"
        ]
    )
)

idea_features = pd.read_parquet(
    stage1_file_paths[
        "idea_features"
    ]
)

In [8]:
stage1_table_summary = pd.DataFrame(
    {
        "table": [
            "trades",
            "trades_with_ideas",
            "trades_with_previous_completed",
            "idea_features",
        ],
        "rows": [
            len(trades),
            len(trades_with_ideas),
            len(
                trades_with_previous_completed
            ),
            len(idea_features),
        ],
        "columns": [
            trades.shape[1],
            trades_with_ideas.shape[1],
            trades_with_previous_completed.shape[1],
            idea_features.shape[1],
        ],
    }
)

display(stage1_table_summary)

,table,rows,columns
0,trades,46520,35
1,trades_with_ideas,46520,69
2,trades_with_previous_completed,46520,42
3,idea_features,33379,34


# Create trade-level features table

In [9]:
trade_features_table = (
    trades_with_ideas.copy()
)

In [10]:
previous_completed_columns = [
    "trade_row_id",
    "previous_completed_close_date_time",
    "previous_completed_net_profit",
    "previous_completed_amount",
    "previous_completed_position_id",
    "previous_completed_was_loss",
    "previous_completed_was_win",
    "reentry_gap_minutes",
]

trade_features_table = (
    trade_features_table
    .merge(
        trades_with_previous_completed[
            previous_completed_columns
        ],
        on="trade_row_id",
        how="left",
        validate="one_to_one",
    )
)

In [11]:
trade_features_table[
    "current_to_previous_amount_ratio"
] = (
    trade_features_table["amount"]
    / trade_features_table[
        "previous_completed_amount"
    ]
)

In [12]:
trade_features_table[
    "current_to_previous_amount_ratio"
] = (
    trade_features_table[
        "current_to_previous_amount_ratio"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

In [13]:
trade_features_table = (
    trade_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "trade_row_id",
        ]
    )
    .reset_index(drop=True)
)

In [14]:
previous_completed_time_valid = (
    trade_features_table[
        "previous_completed_close_date_time"
    ].isna()
    | (
        trade_features_table[
            "previous_completed_close_date_time"
        ]
        < trade_features_table[
            "open_date_time"
        ]
    )
)

print(
    "Previous completed trades are strictly pre-trade:",
    previous_completed_time_valid.all(),
)

Previous completed trades are strictly pre-trade: False


# Create idea-level features table

In [15]:
idea_features_table = (
    idea_features
    .copy()
)

idea_features_table = (
    idea_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "idea_start_time",
            "idea_end_time",
            "idea_id",
        ]
    )
    .reset_index(drop=True)
)

# reverseProfit-per-lot target

In [16]:
# Validate amount before calculating the target.
print(
    "Rows with missing amount:",
    trades["amount"].isna().sum(),
)

print(
    "Rows with non-positive amount:",
    trades["amount"].le(0).sum(),
)

print(
    "Rows with missing reverse profit:",
    trades["reverse_profit"].isna().sum(),
)

Rows with missing amount: 0
Rows with non-positive amount: 0
Rows with missing reverse profit: 0


In [17]:
trades[
    "reverse_profit_per_lot"
] = (
    trades["reverse_profit"]
    / trades["amount"]
)

In [18]:
print(
    "Missing reverse profit per lot:",
    trades[
        "reverse_profit_per_lot"
    ].isna().sum(),
)

print(
    "Infinite reverse profit per lot:",
    np.isinf(
        trades[
            "reverse_profit_per_lot"
        ]
    ).sum(),
)

Missing reverse profit per lot: 0
Infinite reverse profit per lot: 0


In [19]:
display(
    trades[
        "reverse_profit_per_lot"
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

count    46520.000000
mean        -5.372714
std        932.850379
min     -11941.000000
1%       -2664.860000
5%       -1205.000000
25%       -317.000000
50%        -33.000000
75%        329.000000
95%       1204.000000
99%       2642.000000
max      20565.000000
Name: reverse_profit_per_lot, dtype: float64

In [20]:
trade_features_table = (
    trade_features_table
    .merge(
        trades[
            [
                "trade_row_id",
                "reverse_profit_per_lot",
            ]
        ],
        on="trade_row_id",
        how="left",
        validate="one_to_one",
    )
)

In [21]:
assert (
    len(trade_features_table)
    == len(trades)
)

assert (
    trade_features_table[
        "trade_row_id"
    ].is_unique
)

print(
    "Missing target after merge:",
    trade_features_table[
        "reverse_profit_per_lot"
    ].isna().sum(),
)

Missing target after merge: 0


# Pass/breach label

In [22]:
import numpy as np
import pandas as pd

STARTING_BALANCE = 5_000.0
PROFIT_TARGET_RATE = 0.08
MIN_UNIQUE_TRADE_IDEAS = 5
MIN_TRADING_DAYS = 1
ANTI_MARTINGALE_MULTIPLIER = 2.5
PROFIT_DISTRIBUTION_CAP = 0.30

In [23]:
idea_summary = (
    trade_features_table
    .groupby(
        [
            "account_id",
            "campaign_id",
            "idea_id",
        ],
        as_index=False,
    )
    .agg(
        idea_start_time=(
            "open_date_time",
            "min",
        ),
        idea_end_time=(
            "close_date_time",
            "max",
        ),
        side=(
            "side",
            "first",
        ),
        idea_net_profit=(
            "net_profit",
            "sum",
        ),
        idea_trade_count=(
            "trade_row_id",
            "count",
        ),
    )
)

In [24]:
account_campaign_summary = (
    trade_features_table
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        total_net_profit=(
            "net_profit",
            "sum",
        ),
        total_reverse_profit=(
            "reverse_profit",
            "sum",
        ),
        trade_count=(
            "trade_row_id",
            "count",
        ),
        unique_trade_ideas=(
            "idea_id",
            "nunique",
        ),
        first_open_date_time=(
            "open_date_time",
            "min",
        ),
        last_close_date_time=(
            "close_date_time",
            "max",
        ),
        average_amount=(
            "amount",
            "mean",
        ),
        minimum_amount=(
            "amount",
            "min",
        ),
        maximum_amount=(
            "amount",
            "max",
        ),
    )
)

account_campaign_summary[
    "trading_days"
] = (
    trade_features_table
    .assign(
        trade_date=lambda data: (
            data["open_date_time"].dt.date
        )
    )
    .groupby(
        [
            "account_id",
            "campaign_id",
        ]
    )["trade_date"]
    .nunique()
    .values
)

account_campaign_summary[
    "profit_target_amount"
] = (
    STARTING_BALANCE
    * PROFIT_TARGET_RATE
)

account_campaign_summary[
    "meets_profit_target"
] = (
    account_campaign_summary[
        "total_net_profit"
    ]
    >= account_campaign_summary[
        "profit_target_amount"
    ]
)

account_campaign_summary[
    "meets_min_unique_trade_ideas"
] = (
    account_campaign_summary[
        "unique_trade_ideas"
    ]
    >= MIN_UNIQUE_TRADE_IDEAS
)

account_campaign_summary[
    "meets_min_trading_days"
] = (
    account_campaign_summary[
        "trading_days"
    ]
    >= MIN_TRADING_DAYS
)

## ATLSR breach

In [25]:
trade_amount_with_avg = (
    trade_features_table
    .merge(
        account_campaign_summary[
            [
                "account_id",
                "campaign_id",
                "average_amount",
                "minimum_amount",
                "maximum_amount",
            ]
        ],
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="many_to_one",
    )
)

trade_amount_with_avg[
    "atlsr_band_violation"
] = (
    trade_amount_with_avg["amount"]
    > (
        2.0
        * trade_amount_with_avg[
            "average_amount"
        ]
    )
) | (
    trade_amount_with_avg["amount"]
    < (
        trade_amount_with_avg[
            "average_amount"
        ]
        / 4.0
    )
)

atlsr_summary = (
    trade_amount_with_avg
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        any_atlsr_band_violation=(
            "atlsr_band_violation",
            "any",
        )
    )
)

atlsr_summary = (
    atlsr_summary
    .merge(
        account_campaign_summary[
            [
                "account_id",
                "campaign_id",
                "minimum_amount",
                "maximum_amount",
            ]
        ],
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

atlsr_summary[
    "atlsr_highest_vs_lowest_violation"
] = (
    atlsr_summary["maximum_amount"]
    >= (
        2.0
        * atlsr_summary["minimum_amount"]
    )
)

atlsr_summary[
    "breach_atlsr"
] = (
    atlsr_summary[
        "any_atlsr_band_violation"
    ]
    | atlsr_summary[
        "atlsr_highest_vs_lowest_violation"
    ]
)

## Anti-martingale breach

In [26]:
anti_martingale_trades = (
    trade_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "trade_row_id",
        ]
    )
    .copy()
)

group_keys = [
    "account_id",
    "campaign_id",
]

anti_martingale_trades[
    "prior_trade_count"
] = (
    anti_martingale_trades
    .groupby(group_keys)
    .cumcount()
)

anti_martingale_trades[
    "cumulative_amount"
] = (
    anti_martingale_trades
    .groupby(group_keys)["amount"]
    .cumsum()
)

anti_martingale_trades[
    "prior_amount_sum"
] = (
    anti_martingale_trades[
        "cumulative_amount"
    ]
    - anti_martingale_trades["amount"]
)

anti_martingale_trades[
    "prior_average_amount"
] = np.where(
    anti_martingale_trades[
        "prior_trade_count"
    ] > 0,
    (
        anti_martingale_trades[
            "prior_amount_sum"
        ]
        / anti_martingale_trades[
            "prior_trade_count"
        ]
    ),
    np.nan,
)

anti_martingale_trades[
    "anti_martingale_violation"
] = (
    anti_martingale_trades[
        "prior_trade_count"
    ].gt(0)
    & (
        anti_martingale_trades["amount"]
        > (
            ANTI_MARTINGALE_MULTIPLIER
            * anti_martingale_trades[
                "prior_average_amount"
            ]
        )
    )
)

anti_martingale_summary = (
    anti_martingale_trades
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        breach_anti_martingale=(
            "anti_martingale_violation",
            "any",
        )
    )
)

## Profit distribution breach

In [27]:
profit_distribution_summary = (
    idea_summary
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        maximum_idea_profit=(
            "idea_net_profit",
            "max",
        )
    )
)

profit_distribution_summary = (
    profit_distribution_summary
    .merge(
        account_campaign_summary[
            [
                "account_id",
                "campaign_id",
                "total_net_profit",
            ]
        ],
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

profit_distribution_summary[
    "breach_profit_distribution"
] = (
    profit_distribution_summary[
        "total_net_profit"
    ].gt(0)
    & (
        profit_distribution_summary[
            "maximum_idea_profit"
        ]
        > (
            PROFIT_DISTRIBUTION_CAP
            * profit_distribution_summary[
                "total_net_profit"
            ]
        )
    )
)

## Grid/averaging down soft breach

In [28]:
grid_trades = (
    trade_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "side",
            "open_date_time",
            "trade_row_id",
        ]
    )
    .copy()
)

grid_group_columns = [
    "account_id",
    "campaign_id",
    "side",
]

grid_trades[
    "previous_open_date_time"
] = (
    grid_trades
    .groupby(grid_group_columns)[
        "open_date_time"
    ]
    .shift(1)
)

grid_trades[
    "previous_open_price"
] = (
    grid_trades
    .groupby(grid_group_columns)[
        "open_price"
    ]
    .shift(1)
)

grid_trades[
    "seconds_since_previous_same_side_entry"
] = (
    (
        grid_trades["open_date_time"]
        - grid_trades[
            "previous_open_date_time"
        ]
    )
    .dt.total_seconds()
)

In [29]:
grid_trades[
    "worse_price_than_previous"
] = np.select(
    [
        grid_trades["side"].str.lower().eq("buy"),
        grid_trades["side"].str.lower().eq("sell"),
    ],
    [
        (
            grid_trades["open_price"]
            < grid_trades[
                "previous_open_price"
            ]
        ),
        (
            grid_trades["open_price"]
            > grid_trades[
                "previous_open_price"
            ]
        ),
    ],
    default=False,
)

In [30]:
grid_trades[
    "grid_step"
] = (
    grid_trades[
        "seconds_since_previous_same_side_entry"
    ]
    .le(30)
    & grid_trades[
        "worse_price_than_previous"
    ]
)

In [31]:
grid_trades[
    "previous_grid_step"
] = (
    grid_trades
    .groupby(grid_group_columns)[
        "grid_step"
    ]
    .shift(1)
    .fillna(False)
)

grid_trades[
    "grid_soft_breach_trade"
] = (
    grid_trades["grid_step"]
    & grid_trades[
        "previous_grid_step"
    ]
)

In [32]:
grid_breach_summary = (
    grid_trades
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        breach_grid_soft=(
            "grid_soft_breach_trade",
            "any",
        )
    )
)

## Combine pass/breach labels

In [33]:
account_campaign_labels = (
    account_campaign_summary
    .merge(
        atlsr_summary[
            [
                "account_id",
                "campaign_id",
                "breach_atlsr",
            ]
        ],
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        anti_martingale_summary,
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        profit_distribution_summary[
            [
                "account_id",
                "campaign_id",
                "breach_profit_distribution",
            ]
        ],
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        grid_breach_summary,
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

In [34]:
account_campaign_labels[
    "observable_hard_breach"
] = (
    account_campaign_labels[
        "breach_atlsr"
    ]
    | account_campaign_labels[
        "breach_anti_martingale"
    ]
    | account_campaign_labels[
        "breach_profit_distribution"
    ]
)

In [35]:
breach_columns = [
    "breach_atlsr",
    "breach_anti_martingale",
    "breach_profit_distribution",
    "breach_grid_soft",
]

account_campaign_labels[
    breach_columns
] = (
    account_campaign_labels[
        breach_columns
    ]
    .fillna(False)
    .astype(bool)
)

In [36]:
account_campaign_labels[
    "observable_pass"
] = (
    account_campaign_labels[
        "meets_profit_target"
    ]
    & account_campaign_labels[
        "meets_min_unique_trade_ideas"
    ]
    & account_campaign_labels[
        "meets_min_trading_days"
    ]
    & ~account_campaign_labels[
        "observable_hard_breach"
    ]
)

In [37]:
account_campaign_labels[
    "observable_status"
] = np.select(
    [
        account_campaign_labels[
            "observable_hard_breach"
        ],
        account_campaign_labels[
            "observable_pass"
        ],
        account_campaign_labels[
            "breach_grid_soft"
        ],
    ],
    [
        "hard_breach",
        "pass",
        "soft_breach_only",
    ],
    default="neither",
)

In [38]:
label_columns = [
    "account_id",
    "campaign_id",
    "total_net_profit",
    "profit_target_amount",
    "unique_trade_ideas",
    "trading_days",
    "meets_profit_target",
    "meets_min_unique_trade_ideas",
    "meets_min_trading_days",
    "breach_atlsr",
    "breach_anti_martingale",
    "breach_profit_distribution",
    "breach_grid_soft",
    "observable_hard_breach",
    "observable_pass",
    "observable_status",
]

display(
    account_campaign_labels[
        label_columns
    ].head(30)
)

,account_id,campaign_id,total_net_profit,profit_target_amount,unique_trade_ideas,trading_days,meets_profit_target,meets_min_unique_trade_ideas,meets_min_trading_days,breach_atlsr,breach_anti_martingale,breach_profit_distribution,breach_grid_soft,observable_hard_breach,observable_pass,observable_status
0,D#1589378,33,0.01,400.0,1,1,False,False,True,False,False,True,False,True,False,hard_breach
1,D#1589378,37,403.10,400.0,7,1,True,True,True,False,False,True,False,True,False,hard_breach
2,D#1589378,42,437.23,400.0,6,1,True,True,True,True,True,True,False,True,False,hard_breach
3,D#1589378,43,409.10,400.0,9,1,True,True,True,True,False,False,False,True,False,hard_breach
4,D#1589378,45,-212.54,400.0,4,1,False,False,True,False,False,False,False,False,False,neither
5,D#1589378,46,-350.65,400.0,3,1,False,False,True,True,True,False,False,True,False,hard_breach
6,D#1589378,56,-200.80,400.0,5,1,False,True,True,True,False,False,False,True,False,hard_breach
7,D#1589378,60,49.10,400.0,2,1,False,False,True,False,False,True,False,True,False,hard_breach
8,D#1589378,61,-203.20,400.0,1,1,False,False,True,False,False,False,False,False,False,neither
9,D#1589378,62,-184.26,400.0,10,1,False,True,True,True,True,False,False,True,False,hard_breach


In [39]:
display(
    account_campaign_labels[
        "observable_status"
    ]
    .value_counts(
        dropna=False
    )
)

observable_status
hard_breach         4472
neither             3559
pass                 130
soft_breach_only       4
Name: count, dtype: int64

In [40]:
print(
    "Profit target amount:",
    account_campaign_labels[
        "profit_target_amount"
    ].unique(),
)

print(
    "Number of observable passes:",
    account_campaign_labels[
        "observable_pass"
    ].sum(),
)

print(
    "Number of observable hard breaches:",
    account_campaign_labels[
        "observable_hard_breach"
    ].sum(),
)

print(
    "Number of grid soft breaches:",
    account_campaign_labels[
        "breach_grid_soft"
    ].sum(),
)

Profit target amount: [400.]
Number of observable passes: 130
Number of observable hard breaches: 4472
Number of grid soft breaches: 19


# Post-loss behaviour

## Identify post-loss entries

In [41]:
trade_features_table[
    "post_loss_entry"
] = (
    trade_features_table[
        "previous_completed_was_loss"
    ]
    .fillna(False)
    .astype(bool)
)

## Post-loss re-entry speed

In [42]:
trade_features_table[
    "post_loss_reentry_gap_minutes"
] = (
    trade_features_table[
        "reentry_gap_minutes"
    ]
    .where(
        trade_features_table[
            "post_loss_entry"
        ]
    )
)

## Post-loss position-size change

In [43]:
trade_features_table[
    "post_loss_amount_ratio"
] = (
    trade_features_table[
        "current_to_previous_amount_ratio"
    ]
    .where(
        trade_features_table[
            "post_loss_entry"
        ]
    )
)

In [44]:
trade_features_table[
    "post_loss_amount_increased"
] = (
    trade_features_table[
        "post_loss_entry"
    ]
    & (
        trade_features_table[
            "post_loss_amount_ratio"
        ]
        > 1
    )
)

## Add availability

In [45]:
trade_features_table[
    "post_loss_features_available"
] = (
    trade_features_table[
        "post_loss_entry"
    ]
    & trade_features_table[
        "post_loss_reentry_gap_minutes"
    ].notna()
    & trade_features_table[
        "post_loss_amount_ratio"
    ].notna()
)

## Basic validation

In [46]:
post_loss_gap_valid = (
    trade_features_table.loc[
        trade_features_table[
            "post_loss_reentry_gap_minutes"
        ].notna(),
        "post_loss_reentry_gap_minutes",
    ]
    .ge(0)
    .all()
)

print(
    "Post-loss re-entry gaps are non-negative:",
    post_loss_gap_valid,
)

Post-loss re-entry gaps are non-negative: True


In [47]:
post_loss_ratio_valid = (
    trade_features_table.loc[
        trade_features_table[
            "post_loss_amount_ratio"
        ].notna(),
        "post_loss_amount_ratio",
    ]
    .gt(0)
    .all()
)

print(
    "Post-loss amount ratios are positive:",
    post_loss_ratio_valid,
)

Post-loss amount ratios are positive: True


In [48]:
non_post_loss_values_missing = (
    trade_features_table.loc[
        ~trade_features_table[
            "post_loss_entry"
        ],
        [
            "post_loss_reentry_gap_minutes",
            "post_loss_amount_ratio",
        ],
    ]
    .isna()
    .all()
    .all()
)

print(
    "Non-post-loss rows have no post-loss values:",
    non_post_loss_values_missing,
)

Non-post-loss rows have no post-loss values: True


## Inspect actual post-loss rows

In [49]:
post_loss_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "open_date_time",
    "amount",
    "previous_completed_close_date_time",
    "previous_completed_net_profit",
    "previous_completed_amount",
    "post_loss_entry",
    "post_loss_reentry_gap_minutes",
    "post_loss_amount_ratio",
    "post_loss_amount_increased",
    "reverse_profit_per_lot",
]

display(
    trade_features_table.loc[
        trade_features_table[
            "post_loss_entry"
        ],
        post_loss_columns,
    ]
    .head(30)
)

,trade_row_id,account_id,campaign_id,open_date_time,amount,previous_completed_close_date_time,previous_completed_net_profit,previous_completed_amount,post_loss_entry,post_loss_reentry_gap_minutes,post_loss_amount_ratio,post_loss_amount_increased,reverse_profit_per_lot
2,3,D#1589408,33,2026-02-24 02:53:40+00:00,0.10,2026-02-24 02:52:47+00:00,-187.92,0.43,True,0.883333,0.232558,False,-95.0
17,18,D#1645625,33,2026-02-25 00:40:38+00:00,0.03,2026-02-24 13:39:46+00:00,-130.27,0.03,True,660.866667,1.000000,False,-3510.0
24,25,D#1645646,33,2026-02-24 03:19:46+00:00,0.11,2026-02-24 02:33:50+00:00,-36.41,0.07,True,45.933333,1.571429,True,143.0
25,26,D#1645646,33,2026-02-24 03:35:37+00:00,0.11,2026-02-24 03:21:08+00:00,-20.36,0.11,True,14.483333,1.000000,False,112.0
26,27,D#1645646,33,2026-02-24 03:57:12+00:00,0.04,2026-02-24 03:36:14+00:00,-16.95,0.11,True,20.966667,0.363636,False,958.0
27,28,D#1645646,33,2026-02-24 06:21:02+00:00,0.12,2026-02-24 04:01:44+00:00,-40.00,0.04,True,139.300000,3.000000,True,307.0
28,29,D#1645646,33,2026-02-24 06:21:40+00:00,0.07,2026-02-24 06:21:35+00:00,-41.88,0.12,True,0.083333,0.583333,False,-829.0
32,33,D#1645646,33,2026-02-24 10:35:49+00:00,0.05,2026-02-24 10:16:15+00:00,-5.36,0.05,True,19.566667,1.000000,False,-787.0
34,35,D#1645646,33,2026-02-24 17:58:28+00:00,0.05,2026-02-24 12:00:00+00:00,-18.82,0.11,True,358.466667,0.454545,False,-15.0
35,36,D#1645646,33,2026-02-24 17:58:39+00:00,0.12,2026-02-24 17:58:34+00:00,-1.36,0.05,True,0.083333,2.400000,True,-312.0


In [50]:
total_trades = len(
    trade_features_table
)

post_loss_trades = (
    trade_features_table[
        "post_loss_entry"
    ].sum()
)

available_post_loss_trades = (
    trade_features_table[
        "post_loss_features_available"
    ].sum()
)

size_increase_after_loss_trades = (
    trade_features_table[
        "post_loss_amount_increased"
    ].sum()
)

print(
    "Total trades:",
    total_trades,
)

print(
    "Post-loss trades:",
    post_loss_trades,
)

print(
    "Post-loss trades with both features available:",
    available_post_loss_trades,
)

print(
    "Post-loss trades with increased size:",
    size_increase_after_loss_trades,
)

print(
    "Post-loss share of all trades:",
    post_loss_trades / total_trades,
)

Total trades: 46520
Post-loss trades: 18329
Post-loss trades with both features available: 18329
Post-loss trades with increased size: 4034
Post-loss share of all trades: 0.3940025795356836


In [51]:
display(
    trade_features_table.loc[
        trade_features_table[
            "post_loss_features_available"
        ],
        [
            "post_loss_reentry_gap_minutes",
            "post_loss_amount_ratio",
            "reverse_profit_per_lot",
        ],
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

,post_loss_reentry_gap_minutes,post_loss_amount_ratio,reverse_profit_per_lot
count,18329.000000,18329.000000,18329.000000
mean,23.046684,1.496841,-1.720354
std,74.145121,3.026564,819.011232
min,0.000000,0.016667,-10200.000000
1%,0.016667,0.300000,-2696.880000
5%,0.050000,0.857143,-1205.000000
10%,0.083333,1.000000,-783.200000
25%,0.233333,1.000000,-261.000000
50%,1.216667,1.000000,44.000000
75%,8.266667,1.000000,336.000000


## Walk-forward testing

### Prepare test data

In [52]:
# Only rows where the post-loss feature and target are available.
post_loss_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "post_loss_features_available"
        ]
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "post_loss_reentry_gap_minutes",
            "post_loss_amount_ratio",
            "post_loss_amount_increased",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

print(
    "Eligible post-loss trades:",
    len(post_loss_test_data),
)

print(
    "Campaigns represented:",
    post_loss_test_data[
        "campaign_id"
    ].nunique(),
)

Eligible post-loss trades: 18329
Campaigns represented: 34


### Learn fast re-entry threshold function

In [53]:
def learn_fast_reentry_threshold(
    train_data: pd.DataFrame,
    gap_column: str = "post_loss_reentry_gap_minutes",
    target_column: str = "reverse_profit_per_lot",
    candidate_quantiles: tuple[float, ...] = (
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a fast-reentry threshold from historical campaigns.

    Candidate thresholds are derived only from the training data.
    The selected threshold maximizes mean reverse-profit-per-lot
    uplift between fast-reentry and slower-reentry trades.

    Args:
        train_data: Historical post-loss trade observations.
        gap_column: Post-loss re-entry gap feature.
        target_column: Reverse-profit-per-lot evaluation target.
        candidate_quantiles: Training-data quantiles considered as
            possible fast-reentry thresholds.
        min_signal_trades: Minimum number of trades below a threshold.
        min_control_trades: Minimum number above a threshold.

    Returns:
        Dictionary describing the best threshold, or None when no
        candidate has sufficient observations.
    """
    valid_data = (
        train_data[
            [
                gap_column,
                target_column,
            ]
        ]
        .dropna()
        .copy()
    )

    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = (
            valid_data[
                gap_column
            ]
            .quantile(quantile)
        )

        signal_mask = (
            valid_data[
                gap_column
            ]
            <= threshold
        )

        signal_data = valid_data.loc[
            signal_mask
        ]

        control_data = valid_data.loc[
            ~signal_mask
        ]

        if (
            len(signal_data) < min_signal_trades
            or len(control_data) < min_control_trades
        ):
            continue

        signal_mean = (
            signal_data[
                target_column
            ].mean()
        )

        control_mean = (
            control_data[
                target_column
            ].mean()
        )

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold_minutes": threshold,
                "signal_trades": len(
                    signal_data
                ),
                "control_trades": len(
                    control_data
                ),
                "signal_mean": signal_mean,
                "control_mean": control_mean,
                "training_uplift": (
                    signal_mean
                    - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: (
            result[
                "training_uplift"
            ]
        ),
    )

### Re-entry gap

In [54]:
def walk_forward_post_loss_reentry(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests fast post-loss re-entry by campaign.

    For every test campaign, the fast-reentry threshold is learned
    only from earlier campaigns.

    Args:
        data: Post-loss feature-check dataset.
        minimum_training_campaigns: Number of historical campaigns
            required before the first test campaign.

    Returns:
        One row per walk-forward test campaign.
    """
    campaign_order = (
        data[
            "campaign_id"
        ]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[
                :test_index
            ]
        )

        test_campaign = (
            campaign_order[
                test_index
            ]
        )

        train_data = data.loc[
            data[
                "campaign_id"
            ].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data[
                "campaign_id"
            ].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_fast_reentry_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = (
            threshold_result[
                "threshold_minutes"
            ]
        )

        test_data[
            "fast_reentry_signal"
        ] = (
            test_data[
                "post_loss_reentry_gap_minutes"
            ]
            <= threshold
        )

        signal_data = (
            test_data.loc[
                test_data[
                    "fast_reentry_signal"
                ]
            ]
        )

        control_data = (
            test_data.loc[
                ~test_data[
                    "fast_reentry_signal"
                ]
            ]
        )

        if (
            signal_data.empty
            or control_data.empty
        ):
            continue

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        baseline_mean = (
            test_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        fold_results.append(
            {
                "test_campaign": (
                    test_campaign
                ),
                "training_campaign_count": (
                    len(
                        training_campaigns
                    )
                ),
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold_minutes": (
                    threshold
                ),
                "training_uplift": (
                    threshold_result[
                        "training_uplift"
                    ]
                ),
                "test_signal_trades": (
                    len(
                        signal_data
                    )
                ),
                "test_control_trades": (
                    len(
                        control_data
                    )
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "all_post_loss_mean_reverse_profit_per_lot": (
                    baseline_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean
                    - control_mean
                ),
                "test_uplift_vs_all_post_loss": (
                    signal_mean
                    - baseline_mean
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [55]:
post_loss_reentry_walk_forward = (
    walk_forward_post_loss_reentry(
        post_loss_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    post_loss_reentry_walk_forward
)

,test_campaign,training_campaign_count,learned_quantile,learned_threshold_minutes,training_uplift,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,all_post_loss_mean_reverse_profit_per_lot,test_uplift_vs_control,test_uplift_vs_all_post_loss
0,43,10,0.5,1.033333,0.094244,270,282,7.254321,73.321986,41.006280,-66.067665,-33.751959
1,44,11,0.5,1.033333,-7.034557,347,295,-3.328530,-35.929153,-18.308567,32.600622,14.980037
2,45,12,0.1,0.083333,2.170769,44,411,-67.172004,14.514645,6.615277,-81.686649,-73.787281
3,46,13,0.5,1.033333,5.054154,294,256,-12.988095,-31.725977,-21.709727,18.737881,8.721632
4,47,14,0.1,0.066667,37.245406,21,348,71.142857,-31.064725,-25.248033,102.207582,96.390890
5,48,15,0.1,0.083333,6.207736,74,480,18.824324,-10.265625,-6.379964,29.089949,25.204288
6,49,16,0.1,0.083333,7.913970,52,463,80.615385,-37.283458,-25.379109,117.898843,105.994493
7,50,17,0.1,0.083333,13.869654,59,417,-49.112712,71.771543,56.787990,-120.884255,-105.900702
8,51,18,0.1,0.083333,6.597995,42,408,73.738095,12.136169,17.885683,61.601926,55.852413
9,52,19,0.1,0.083333,8.444068,31,354,69.161290,22.138418,25.924675,47.022872,43.236615


### Display results

In [56]:
display(
    post_loss_reentry_walk_forward[
        [
            "test_campaign",
            "learned_threshold_minutes",
            "test_signal_trades",
            "signal_mean_reverse_profit_per_lot",
            "control_mean_reverse_profit_per_lot",
            "test_uplift_vs_control",
        ]
    ]
)

,test_campaign,learned_threshold_minutes,test_signal_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,test_uplift_vs_control
0,43,1.033333,270,7.254321,73.321986,-66.067665
1,44,1.033333,347,-3.328530,-35.929153,32.600622
2,45,0.083333,44,-67.172004,14.514645,-81.686649
3,46,1.033333,294,-12.988095,-31.725977,18.737881
4,47,0.066667,21,71.142857,-31.064725,102.207582
5,48,0.083333,74,18.824324,-10.265625,29.089949
6,49,0.083333,52,80.615385,-37.283458,117.898843
7,50,0.083333,59,-49.112712,71.771543,-120.884255
8,51,0.083333,42,73.738095,12.136169,61.601926
9,52,0.083333,31,69.161290,22.138418,47.022872


In [57]:
print(
    "Number of walk-forward folds:",
    len(
        post_loss_reentry_walk_forward
    ),
)

print(
    "Mean OOS uplift vs control:",
    post_loss_reentry_walk_forward[
        "test_uplift_vs_control"
    ].mean(),
)

print(
    "Median OOS uplift vs control:",
    post_loss_reentry_walk_forward[
        "test_uplift_vs_control"
    ].median(),
)

print(
    "Positive-uplift campaign share:",
    (
        post_loss_reentry_walk_forward[
            "test_uplift_vs_control"
        ]
        > 0
    ).mean(),
)

Number of walk-forward folds: 24
Mean OOS uplift vs control: -1.1693560451996134
Median OOS uplift vs control: 10.427380975957425
Positive-uplift campaign share: 0.5833333333333334


### Pool the actual out-of-sample selected trades

In [58]:
def collect_walk_forward_signals(
    data: pd.DataFrame,
    walk_forward_results: pd.DataFrame,
) -> pd.DataFrame:
    """Collects trades selected by frozen walk-forward thresholds."""
    output = []

    for _, fold in (
        walk_forward_results.iterrows()
    ):
        campaign_data = (
            data.loc[
                data[
                    "campaign_id"
                ].eq(
                    fold[
                        "test_campaign"
                    ]
                )
            ]
            .copy()
        )

        campaign_data[
            "fast_reentry_signal"
        ] = (
            campaign_data[
                "post_loss_reentry_gap_minutes"
            ]
            <= fold[
                "learned_threshold_minutes"
            ]
        )

        campaign_data[
            "walk_forward_threshold_minutes"
        ] = fold[
            "learned_threshold_minutes"
        ]

        output.append(
            campaign_data
        )

    return pd.concat(
        output,
        ignore_index=True,
    )

In [59]:
post_loss_reentry_oos = (
    collect_walk_forward_signals(
        post_loss_test_data,
        post_loss_reentry_walk_forward,
    )
)

In [60]:
oos_signal = (
    post_loss_reentry_oos.loc[
        post_loss_reentry_oos[
            "fast_reentry_signal"
        ]
    ]
)

oos_control = (
    post_loss_reentry_oos.loc[
        ~post_loss_reentry_oos[
            "fast_reentry_signal"
        ]
    ]
)

print(
    "OOS signal trades:",
    len(oos_signal),
)

print(
    "OOS control trades:",
    len(oos_control),
)

print(
    "OOS signal mean reverse profit per lot:",
    oos_signal[
        "reverse_profit_per_lot"
    ].mean(),
)

print(
    "OOS control mean reverse profit per lot:",
    oos_control[
        "reverse_profit_per_lot"
    ].mean(),
)

print(
    "OOS pooled uplift:",
    (
        oos_signal[
            "reverse_profit_per_lot"
        ].mean()
        - oos_control[
            "reverse_profit_per_lot"
        ].mean()
    ),
)

OOS signal trades: 4864
OOS control trades: 8732
OOS signal mean reverse profit per lot: -5.466473919969256
OOS control mean reverse profit per lot: 2.562408286498516
OOS pooled uplift: -8.028882206467772


### Campaign-level 95% confidence interval

In [61]:
def bootstrap_campaign_uplift_ci(
    fold_results: pd.DataFrame,
    uplift_column: str = (
        "test_uplift_vs_control"
    ),
    n_bootstrap: int = 10_000,
    random_state: int = 42,
) -> tuple[float, float]:
    """Bootstraps a 95% CI for mean campaign-level OOS uplift."""
    uplifts = (
        fold_results[
            uplift_column
        ]
        .dropna()
        .to_numpy()
    )

    rng = np.random.default_rng(
        random_state
    )

    bootstrap_means = np.empty(
        n_bootstrap
    )

    for index in range(
        n_bootstrap
    ):
        sample = rng.choice(
            uplifts,
            size=len(uplifts),
            replace=True,
        )

        bootstrap_means[index] = (
            sample.mean()
        )

    lower, upper = np.quantile(
        bootstrap_means,
        [
            0.025,
            0.975,
        ],
    )

    return lower, upper

In [62]:
uplift_ci_lower, uplift_ci_upper = (
    bootstrap_campaign_uplift_ci(
        post_loss_reentry_walk_forward
    )
)

print(
    "Mean campaign OOS uplift:",
    post_loss_reentry_walk_forward[
        "test_uplift_vs_control"
    ].mean(),
)

print(
    "95% bootstrap CI:",
    (
        uplift_ci_lower,
        uplift_ci_upper,
    ),
)

Mean campaign OOS uplift: -1.1693560451996134
95% bootstrap CI: (np.float64(-31.07700217205202), np.float64(26.653174884269863))


### Size escalation

In [63]:
def walk_forward_post_loss_size(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests post-loss size escalation by campaign.

    Args:
        data: Eligible post-loss observations.
        minimum_training_campaigns: Number of historical campaigns
            required before testing begins.

    Returns:
        One row per OOS test campaign.
    """
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        test_campaign = campaign_order[test_index]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        signal_data = test_data.loc[
            test_data[
                "post_loss_amount_increased"
            ]
        ]

        control_data = test_data.loc[
            ~test_data[
                "post_loss_amount_increased"
            ]
        ]

        if (
            signal_data.empty
            or control_data.empty
        ):
            continue

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        baseline_mean = (
            test_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "test_signal_trades": len(
                    signal_data
                ),
                "test_control_trades": len(
                    control_data
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "all_post_loss_mean_reverse_profit_per_lot": (
                    baseline_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean
                    - control_mean
                ),
                "test_uplift_vs_all_post_loss": (
                    signal_mean
                    - baseline_mean
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [64]:
post_loss_size_walk_forward = (
    walk_forward_post_loss_size(
        post_loss_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    post_loss_size_walk_forward
)

,test_campaign,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,all_post_loss_mean_reverse_profit_per_lot,test_uplift_vs_control,test_uplift_vs_all_post_loss
0,43,92,460,59.813043,37.244928,41.006280,22.568116,18.806763
1,44,127,515,12.011811,-25.785631,-18.308567,37.797442,30.320378
2,45,102,353,-26.076789,16.061709,6.615277,-42.138498,-32.692066
3,46,143,407,46.536014,-45.687961,-21.709727,92.223975,68.245741
4,47,91,278,-3.680486,-32.307914,-25.248033,28.627427,21.567547
5,48,104,450,-4.389423,-6.840000,-6.379964,2.450577,1.990541
6,49,128,387,-109.884696,2.571059,-25.379109,-112.455755,-84.505587
7,50,102,374,-132.116667,108.307442,56.787990,-240.424109,-188.904657
8,51,92,358,-138.873913,58.170271,17.885683,-197.044184,-156.759596
9,52,104,281,91.000000,1.839858,25.924675,89.160142,65.075325


In [65]:
print(
    "Mean OOS uplift vs control:",
    post_loss_size_walk_forward[
        "test_uplift_vs_control"
    ].mean(),
)

print(
    "Median OOS uplift vs control:",
    post_loss_size_walk_forward[
        "test_uplift_vs_control"
    ].median(),
)

print(
    "Positive-uplift campaign share:",
    (
        post_loss_size_walk_forward[
            "test_uplift_vs_control"
        ]
        > 0
    ).mean(),
)

Mean OOS uplift vs control: 0.698648675035173
Median OOS uplift vs control: 18.493087588177183
Positive-uplift campaign share: 0.625


In [66]:
size_ci_lower, size_ci_upper = (
    bootstrap_campaign_uplift_ci(
        post_loss_size_walk_forward
    )
)

print(
    "95% bootstrap CI:",
    (
        size_ci_lower,
        size_ci_upper,
    ),
)

95% bootstrap CI: (np.float64(-38.77729923053847), np.float64(36.474286673327896))


### Fast re-entry + size escalation

In [67]:
def walk_forward_post_loss_combination(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Tests fast post-loss re-entry combined with size escalation.

    The fast-reentry threshold is learned using earlier campaigns only.
    The OOS signal requires both fast re-entry and increased position size.

    Args:
        data: Eligible post-loss observations.
        minimum_training_campaigns: Number of historical campaigns
            required before testing begins.

    Returns:
        One row per OOS test campaign.
    """
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_fast_reentry_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = (
            threshold_result[
                "threshold_minutes"
            ]
        )

        test_data[
            "fast_reentry_signal"
        ] = (
            test_data[
                "post_loss_reentry_gap_minutes"
            ]
            <= threshold
        )

        test_data[
            "fast_and_larger_signal"
        ] = (
            test_data[
                "fast_reentry_signal"
            ]
            & test_data[
                "post_loss_amount_increased"
            ]
        )

        signal_data = test_data.loc[
            test_data[
                "fast_and_larger_signal"
            ]
        ]

        control_data = test_data.loc[
            ~test_data[
                "fast_and_larger_signal"
            ]
        ]

        if (
            signal_data.empty
            or control_data.empty
        ):
            continue

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        baseline_mean = (
            test_data[
                "reverse_profit_per_lot"
            ].mean()
        )

        fold_results.append(
            {
                "test_campaign": (
                    test_campaign
                ),
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold_minutes": (
                    threshold
                ),
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "all_post_loss_mean_reverse_profit_per_lot": (
                    baseline_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean
                    - control_mean
                ),
                "test_uplift_vs_all_post_loss": (
                    signal_mean
                    - baseline_mean
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [68]:
post_loss_combination_walk_forward = (
    walk_forward_post_loss_combination(
        post_loss_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    post_loss_combination_walk_forward
)

,test_campaign,learned_quantile,learned_threshold_minutes,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,all_post_loss_mean_reverse_profit_per_lot,test_uplift_vs_control,test_uplift_vs_all_post_loss
0,43,0.5,1.033333,42,510,-20.857143,46.100915,41.006280,-66.958058,-61.863423
1,44,0.5,1.033333,74,568,-28.527027,-16.977289,-18.308567,-11.549738,-10.218460
2,45,0.1,0.083333,11,444,-241.983471,12.774255,6.615277,-254.757726,-248.598748
3,46,0.5,1.033333,78,472,31.993590,-30.584428,-21.709727,62.578018,53.703317
4,47,0.1,0.066667,5,364,201.400000,-28.361330,-25.248033,229.761330,226.648033
5,48,0.1,0.083333,9,545,77.333333,-7.762385,-6.379964,85.095719,83.713297
6,49,0.1,0.083333,8,507,-14.125000,-25.556688,-25.379109,11.431688,11.254109
7,50,0.1,0.083333,6,470,295.500000,53.740603,56.787990,241.759397,238.712010
8,51,0.1,0.083333,7,443,-42.428571,18.838729,17.885683,-61.267301,-60.314254
9,52,0.1,0.083333,8,377,6.250000,26.342175,25.924675,-20.092175,-19.674675


In [69]:
print(
    "Mean OOS uplift vs control:",
    post_loss_combination_walk_forward[
        "test_uplift_vs_control"
    ].mean(),
)

print(
    "Median OOS uplift vs control:",
    post_loss_combination_walk_forward[
        "test_uplift_vs_control"
    ].median(),
)

print(
    "Positive-uplift campaign share:",
    (
        post_loss_combination_walk_forward[
            "test_uplift_vs_control"
        ]
        > 0
    ).mean(),
)

Mean OOS uplift vs control: 12.700406163894021
Median OOS uplift vs control: 4.532831635032483
Positive-uplift campaign share: 0.5


In [70]:
combination_ci_lower, combination_ci_upper = (
    bootstrap_campaign_uplift_ci(
        post_loss_combination_walk_forward
    )
)

print(
    "95% bootstrap CI:",
    (
        combination_ci_lower,
        combination_ci_upper,
    ),
)

95% bootstrap CI: (np.float64(-42.360007615500294), np.float64(65.67377364872712))


### Result comparison

In [71]:
post_loss_feature_comparison = pd.DataFrame(
    {
        "feature": [
            "fast_reentry",
            "size_escalation",
            "fast_reentry_and_size_escalation",
        ],
        "mean_oos_uplift": [
            post_loss_reentry_walk_forward[
                "test_uplift_vs_control"
            ].mean(),
            post_loss_size_walk_forward[
                "test_uplift_vs_control"
            ].mean(),
            post_loss_combination_walk_forward[
                "test_uplift_vs_control"
            ].mean(),
        ],
        "median_oos_uplift": [
            post_loss_reentry_walk_forward[
                "test_uplift_vs_control"
            ].median(),
            post_loss_size_walk_forward[
                "test_uplift_vs_control"
            ].median(),
            post_loss_combination_walk_forward[
                "test_uplift_vs_control"
            ].median(),
        ],
        "positive_campaign_share": [
            (
                post_loss_reentry_walk_forward[
                    "test_uplift_vs_control"
                ]
                > 0
            ).mean(),
            (
                post_loss_size_walk_forward[
                    "test_uplift_vs_control"
                ]
                > 0
            ).mean(),
            (
                post_loss_combination_walk_forward[
                    "test_uplift_vs_control"
                ]
                > 0
            ).mean(),
        ],
    }
)

display(
    post_loss_feature_comparison
)

,feature,mean_oos_uplift,median_oos_uplift,positive_campaign_share
0,fast_reentry,-1.169356,10.427381,0.583333
1,size_escalation,0.698649,18.493088,0.625000
2,fast_reentry_and_size_escalation,12.700406,4.532832,0.500000


# Holding duration features

## Past losing vs profitable holding duration

### Define attach past duration function

In [72]:
def attach_past_duration_features(
    target_table: pd.DataFrame,
    history_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    history_end_time_column: str,
    duration_column: str,
    profit_column: str,
    feature_level: str,
) -> pd.DataFrame:
    """Attaches past losing and profitable duration features.

    For every target row, only historical records that ended strictly
    before the target timestamp are included.

    Args:
        target_table: Table receiving the historical features.
        history_table: Completed trade-level or idea-level history.
        group_columns: Columns defining independent trading histories.
        target_time_column: Timestamp when the features are measured.
        history_end_time_column: Timestamp when a historical record ended.
        duration_column: Historical holding-duration column.
        profit_column: Historical realized net-profit column.
        feature_level: Name used in generated columns, such as
            `trade` or `idea`.

    Returns:
        A copy of the target table with historical duration features.
    """
    target = target_table.copy()
    history = history_table.copy()

    profitable_duration_column = (
        f"_profitable_{feature_level}_duration"
    )
    losing_duration_column = (
        f"_losing_{feature_level}_duration"
    )

    history[profitable_duration_column] = (
        history[duration_column]
        .where(history[profit_column] > 0)
    )

    history[losing_duration_column] = (
        history[duration_column]
        .where(history[profit_column] < 0)
    )

    profitable_count_column = (
        f"past_profitable_{feature_level}_count"
    )
    losing_count_column = (
        f"past_losing_{feature_level}_count"
    )
    profitable_max_column = (
        f"past_max_profitable_{feature_level}"
        "_duration_minutes"
    )
    losing_max_column = (
        f"past_max_losing_{feature_level}"
        "_duration_minutes"
    )

    output_groups = []

    for group_key, target_group in target.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        history_mask = pd.Series(
            True,
            index=history.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                history_mask &= history[column].isna()
            else:
                history_mask &= history[column].eq(value)

        group_history = (
            history.loc[history_mask]
            .sort_values(
                [
                    history_end_time_column,
                ]
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(target_time_column)
            .copy()
        )

        if group_history.empty:
            current_group[profitable_count_column] = 0
            current_group[losing_count_column] = 0
            current_group[profitable_max_column] = np.nan
            current_group[losing_max_column] = np.nan

            output_groups.append(current_group)
            continue

        group_history[profitable_count_column] = (
            group_history[profit_column]
            .gt(0)
            .astype(int)
            .cumsum()
        )

        group_history[losing_count_column] = (
            group_history[profit_column]
            .lt(0)
            .astype(int)
            .cumsum()
        )

        group_history[profitable_max_column] = (
            group_history[
                profitable_duration_column
            ]
            .cummax()
            .ffill()
        )

        group_history[losing_max_column] = (
            group_history[
                losing_duration_column
            ]
            .cummax()
            .ffill()
        )

        history_merge_time_column = (
            f"_past_{feature_level}_end_time"
        )

        history_for_merge = (
            group_history[
                [
                    history_end_time_column,
                    profitable_count_column,
                    losing_count_column,
                    profitable_max_column,
                    losing_max_column,
                ]
            ]
            .rename(
                columns={
                    history_end_time_column: (
                        history_merge_time_column
                    )
                }
            )
        )

        current_group = pd.merge_asof(
            left=current_group,
            right=history_for_merge,
            left_on=target_time_column,
            right_on=history_merge_time_column,
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            profitable_count_column
        ] = (
            current_group[
                profitable_count_column
            ]
            .fillna(0)
            .astype(int)
        )

        current_group[
            losing_count_column
        ] = (
            current_group[
                losing_count_column
            ]
            .fillna(0)
            .astype(int)
        )

        current_group = current_group.drop(
            columns=[history_merge_time_column],
            errors="ignore",
        )

        output_groups.append(current_group)

    result = pd.concat(
        output_groups,
        ignore_index=True,
    )

    ratio_column = (
        f"past_max_losing_to_profitable_"
        f"{feature_level}_duration_ratio"
    )

    log_ratio_column = (
        f"past_log_max_losing_to_profitable_"
        f"{feature_level}_duration_ratio"
    )

    available_column = (
        f"past_{feature_level}_duration_ratio_available"
    )

    robust_column = (
        f"past_{feature_level}_duration_ratio_robust"
    )

    valid_ratio = (
        result[losing_max_column].gt(0)
        & result[profitable_max_column].gt(0)
    )

    result[ratio_column] = np.where(
        valid_ratio,
        (
            result[losing_max_column]
            / result[profitable_max_column]
        ),
        np.nan,
    )

    result[log_ratio_column] = np.where(
        valid_ratio,
        np.log(result[ratio_column]),
        np.nan,
    )

    result[available_column] = (
        result[losing_count_column].ge(1)
        & result[profitable_count_column].ge(1)
        & valid_ratio
    )

    result[robust_column] = (
        result[losing_count_column].ge(2)
        & result[profitable_count_column].ge(2)
        & valid_ratio
    )

    return result

### Prepare completed trade history

In [73]:
trade_duration_history = (
    trades[
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "open_date_time",
            "close_date_time",
            "duration_sec",
            "net_profit",
        ]
    ]
    .copy()
)

trade_duration_history[
    "trade_duration_minutes"
] = (
    trade_duration_history[
        "duration_sec"
    ]
    / 60
)

### Attach trade-level historical features

In [74]:
trade_features_table = (
    attach_past_duration_features(
        target_table=trade_features_table,
        history_table=trade_duration_history,
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column="open_date_time",
        history_end_time_column="close_date_time",
        duration_column="trade_duration_minutes",
        profit_column="net_profit",
        feature_level="trade",
    )
)

### Restore chronological order

In [75]:
trade_features_table = (
    trade_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "trade_row_id",
        ]
    )
    .reset_index(drop=True)
)

### Inspect trade-level features

In [76]:
trade_duration_feature_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "open_date_time",
    "past_profitable_trade_count",
    "past_losing_trade_count",
    "past_max_profitable_trade_duration_minutes",
    "past_max_losing_trade_duration_minutes",
    "past_max_losing_to_profitable_trade_duration_ratio",
    "past_log_max_losing_to_profitable_trade_duration_ratio",
    "past_trade_duration_ratio_available",
    "past_trade_duration_ratio_robust",
]

display(
    trade_features_table[
        trade_duration_feature_columns
    ].head(20)
)

,trade_row_id,account_id,campaign_id,open_date_time,past_profitable_trade_count,past_losing_trade_count,past_max_profitable_trade_duration_minutes,past_max_losing_trade_duration_minutes,past_max_losing_to_profitable_trade_duration_ratio,past_log_max_losing_to_profitable_trade_duration_ratio,past_trade_duration_ratio_available,past_trade_duration_ratio_robust
0,1,D#1589378,33,2026-02-24 12:01:08+00:00,0,0,NaN,NaN,NaN,NaN,False,False
1,2,D#1589408,33,2026-02-24 02:46:35+00:00,0,0,NaN,NaN,NaN,NaN,False,False
2,3,D#1589408,33,2026-02-24 02:53:40+00:00,0,1,NaN,6.200000,NaN,NaN,False,False
3,4,D#1589408,33,2026-02-24 02:57:37+00:00,1,1,3.150000,6.200000,1.968254,0.677147,True,False
4,5,D#1589410,33,2026-02-25 00:13:04+00:00,0,0,NaN,NaN,NaN,NaN,False,False
5,6,D#1589410,33,2026-02-25 00:23:39+00:00,1,0,8.366667,NaN,NaN,NaN,False,False
6,7,D#1589419,33,2026-02-24 05:32:22+00:00,0,0,NaN,NaN,NaN,NaN,False,False
7,8,D#1589428,33,2026-02-24 02:38:47+00:00,0,0,NaN,NaN,NaN,NaN,False,False
8,9,D#1589428,33,2026-02-24 07:24:19+00:00,1,0,5.366667,NaN,NaN,NaN,False,False
9,10,D#1589428,33,2026-02-24 08:38:11+00:00,2,0,5.483333,NaN,NaN,NaN,False,False


### Attach idea-level historical features

In [77]:
idea_features_table = (
    attach_past_duration_features(
        target_table=idea_features_table,
        history_table=idea_features,
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column="idea_start_time",
        history_end_time_column="idea_end_time",
        duration_column="idea_duration_minutes",
        profit_column="total_net_profit",
        feature_level="idea",
    )
)

### Restore idea order

In [78]:
idea_features_table = (
    idea_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "idea_start_time",
            "idea_end_time",
            "idea_id",
        ]
    )
    .reset_index(drop=True)
)

### Inspect idea-level features

In [79]:
idea_duration_feature_columns = [
    "idea_id",
    "account_id",
    "campaign_id",
    "idea_start_time",
    "past_profitable_idea_count",
    "past_losing_idea_count",
    "past_max_profitable_idea_duration_minutes",
    "past_max_losing_idea_duration_minutes",
    "past_max_losing_to_profitable_idea_duration_ratio",
    "past_log_max_losing_to_profitable_idea_duration_ratio",
    "past_idea_duration_ratio_available",
    "past_idea_duration_ratio_robust",
]

display(
    idea_features_table[
        idea_duration_feature_columns
    ].head(20)
)

,idea_id,account_id,campaign_id,idea_start_time,past_profitable_idea_count,past_losing_idea_count,past_max_profitable_idea_duration_minutes,past_max_losing_idea_duration_minutes,past_max_losing_to_profitable_idea_duration_ratio,past_log_max_losing_to_profitable_idea_duration_ratio,past_idea_duration_ratio_available,past_idea_duration_ratio_robust
0,D#1589378_33_sell_1,D#1589378,33,2026-02-24 12:01:08+00:00,0,0,NaN,NaN,NaN,NaN,False,False
1,D#1589408_33_sell_1,D#1589408,33,2026-02-24 02:46:35+00:00,0,0,NaN,NaN,NaN,NaN,False,False
2,D#1589408_33_buy_1,D#1589408,33,2026-02-24 02:53:40+00:00,0,1,NaN,6.200000,NaN,NaN,False,False
3,D#1589408_33_sell_2,D#1589408,33,2026-02-24 02:57:37+00:00,1,1,3.150000,6.200000,1.968254,0.677147,True,False
4,D#1589410_33_buy_1,D#1589410,33,2026-02-25 00:13:04+00:00,0,0,NaN,NaN,NaN,NaN,False,False
5,D#1589419_33_buy_1,D#1589419,33,2026-02-24 05:32:22+00:00,0,0,NaN,NaN,NaN,NaN,False,False
6,D#1589428_33_buy_1,D#1589428,33,2026-02-24 02:38:47+00:00,0,0,NaN,NaN,NaN,NaN,False,False
7,D#1589428_33_buy_2,D#1589428,33,2026-02-24 07:24:19+00:00,1,0,5.366667,NaN,NaN,NaN,False,False
8,D#1589428_33_buy_3,D#1589428,33,2026-02-24 08:38:11+00:00,2,0,5.483333,NaN,NaN,NaN,False,False
9,D#1614600_33_buy_1,D#1614600,33,2026-02-24 11:26:56+00:00,0,0,NaN,NaN,NaN,NaN,False,False


# Trading activity dynamics

## Cumulative trading intensity

### Define attach past event count function

In [80]:
def attach_past_event_count(
    target_table: pd.DataFrame,
    event_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    event_time_column: str,
    output_column: str,
) -> pd.DataFrame:
    """Attaches the number of past events before each target row.

    Events occurring at exactly the target timestamp are excluded.

    Args:
        target_table: Table receiving the event-count feature.
        event_table: Table containing historical events.
        group_columns: Columns defining independent histories.
        target_time_column: Timestamp at which the count is measured.
        event_time_column: Timestamp when each event occurred.
        output_column: Name of the generated count feature.

    Returns:
        A copy of the target table with the past-event count attached.
    """
    target = target_table.copy()

    event_counts = (
        event_table
        .groupby(
            group_columns + [event_time_column],
            dropna=False,
        )
        .size()
        .rename("_events_at_timestamp")
        .reset_index()
    )

    event_counts = event_counts.sort_values(
        group_columns + [event_time_column]
    )

    event_counts[output_column] = (
        event_counts
        .groupby(
            group_columns,
            dropna=False,
        )["_events_at_timestamp"]
        .cumsum()
    )

    output_groups = []

    for group_key, target_group in target.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        event_mask = pd.Series(
            True,
            index=event_counts.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= event_counts[column].isna()
            else:
                event_mask &= event_counts[column].eq(value)

        group_events = (
            event_counts.loc[
                event_mask,
                [
                    event_time_column,
                    output_column,
                ],
            ]
            .sort_values(event_time_column)
            .copy()
        )

        current_group = (
            target_group
            .sort_values(target_time_column)
            .copy()
        )

        if group_events.empty:
            current_group[output_column] = 0
            output_groups.append(current_group)
            continue

        merge_time_column = (
            f"_past_{output_column}_event_time"
        )

        group_events = group_events.rename(
            columns={
                event_time_column: merge_time_column,
            }
        )

        current_group = pd.merge_asof(
            left=current_group,
            right=group_events,
            left_on=target_time_column,
            right_on=merge_time_column,
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[output_column] = (
            current_group[output_column]
            .fillna(0)
            .astype(int)
        )

        current_group = current_group.drop(
            columns=[merge_time_column],
            errors="ignore",
        )

        output_groups.append(current_group)

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

### Add cumulative trade-opening intensity

In [81]:
trade_features_table = attach_past_event_count(
    target_table=trade_features_table,
    event_table=trades,
    group_columns=[
        "account_id",
        "campaign_id",
    ],
    target_time_column="open_date_time",
    event_time_column="open_date_time",
    output_column="past_trade_open_count",
)

### Add cumulative idea-start intensity

In [82]:
idea_start_events = (
    idea_features[
        [
            "account_id",
            "campaign_id",
            "idea_id",
            "idea_start_time",
        ]
    ]
    .drop_duplicates(subset="idea_id")
    .copy()
)

In [83]:
trade_features_table = attach_past_event_count(
    target_table=trade_features_table,
    event_table=idea_start_events,
    group_columns=[
        "account_id",
        "campaign_id",
    ],
    target_time_column="open_date_time",
    event_time_column="idea_start_time",
    output_column="past_idea_start_count",
)

### Calculate elapsed active time

In [84]:
trade_features_table[
    "first_trade_open_date_time"
] = (
    trade_features_table
    .groupby(
        [
            "account_id",
            "campaign_id",
        ]
    )["open_date_time"]
    .transform("min")
)

In [85]:
trade_features_table[
    "elapsed_active_minutes"
] = (
    trade_features_table["open_date_time"]
    - trade_features_table[
        "first_trade_open_date_time"
    ]
).dt.total_seconds() / 60

In [86]:
trade_features_table[
    "elapsed_active_hours"
] = (
    trade_features_table[
        "elapsed_active_minutes"
    ]
    / 60
)

### Calculate trades opened per active hour

In [87]:
valid_elapsed_time = (
    trade_features_table[
        "elapsed_active_hours"
    ].gt(0)
)

In [88]:
trade_features_table[
    "past_trades_opened_per_active_hour"
] = np.where(
    valid_elapsed_time,
    (
        trade_features_table[
            "past_trade_open_count"
        ]
        / trade_features_table[
            "elapsed_active_hours"
        ]
    ),
    np.nan,
)

### Calculate ideas started per active hour

In [89]:
trade_features_table[
    "past_ideas_started_per_active_hour"
] = np.where(
    valid_elapsed_time,
    (
        trade_features_table[
            "past_idea_start_count"
        ]
        / trade_features_table[
            "elapsed_active_hours"
        ]
    ),
    np.nan,
)

In [90]:
trade_features_table[
    "active_hour_intensity_available"
] = valid_elapsed_time

### Inspect the features

In [91]:
intensity_feature_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "idea_id",
    "open_date_time",
    "past_trade_open_count",
    "past_idea_start_count",
    "elapsed_active_minutes",
    "past_trades_opened_per_active_hour",
    "past_ideas_started_per_active_hour",
]

display(
    trade_features_table[
        intensity_feature_columns
    ].head(30)
)

,trade_row_id,account_id,campaign_id,idea_id,open_date_time,past_trade_open_count,past_idea_start_count,elapsed_active_minutes,past_trades_opened_per_active_hour,past_ideas_started_per_active_hour
0,1,D#1589378,33,D#1589378_33_sell_1,2026-02-24 12:01:08+00:00,0,0,0.000000,NaN,NaN
1,2,D#1589408,33,D#1589408_33_sell_1,2026-02-24 02:46:35+00:00,0,0,0.000000,NaN,NaN
2,3,D#1589408,33,D#1589408_33_buy_1,2026-02-24 02:53:40+00:00,1,1,7.083333,8.470588,8.470588
3,4,D#1589408,33,D#1589408_33_sell_2,2026-02-24 02:57:37+00:00,2,2,11.033333,10.876133,10.876133
4,5,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:13:04+00:00,0,0,0.000000,NaN,NaN
5,6,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:23:39+00:00,1,1,10.583333,5.669291,5.669291
6,7,D#1589419,33,D#1589419_33_buy_1,2026-02-24 05:32:22+00:00,0,0,0.000000,NaN,NaN
7,8,D#1589428,33,D#1589428_33_buy_1,2026-02-24 02:38:47+00:00,0,0,0.000000,NaN,NaN
8,9,D#1589428,33,D#1589428_33_buy_2,2026-02-24 07:24:19+00:00,1,1,285.533333,0.210133,0.210133
9,10,D#1589428,33,D#1589428_33_buy_3,2026-02-24 08:38:11+00:00,2,2,359.400000,0.333890,0.333890


### Walk-forward test

In [92]:
cumulative_intensity_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_trades_opened_per_active_hour"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_trades_opened_per_active_hour",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [93]:
def learn_high_cumulative_intensity_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually high cumulative trading intensity."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "past_trades_opened_per_active_hour"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "past_trades_opened_per_active_hour"
            ]
            >= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [94]:
def walk_forward_high_cumulative_intensity(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually high cumulative trading intensity."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_high_cumulative_intensity_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "past_trades_opened_per_active_hour"
            ]
            >= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": len(
                    signal_data
                ),
                "test_control_trades": len(
                    control_data
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [95]:
cumulative_intensity_walk_forward = (
    walk_forward_high_cumulative_intensity(
        cumulative_intensity_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    cumulative_intensity_walk_forward
)

summarize_walk_forward(
    cumulative_intensity_walk_forward,
    "High cumulative trading intensity",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.9,19.315299,138,1037,-30.601322
1,44,0.9,19.672131,186,1129,22.367917
2,45,0.7,6.202191,288,731,-50.721527
3,46,0.7,6.171309,308,805,44.605008
4,47,0.7,6.088795,199,582,-55.735714
5,48,0.7,6.019689,353,852,-28.719525
6,49,0.9,20.000000,68,1027,-43.974387
7,50,0.9,19.564155,71,846,-90.472337
8,51,0.9,19.426208,67,882,15.130715
9,52,0.9,19.177719,76,714,-24.477235


NameError: name 'summarize_walk_forward' is not defined

## Recent trading intensity

### Define rolling-window event count function

In [ ]:
def attach_rolling_event_counts(
    target_table: pd.DataFrame,
    event_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    event_time_column: str,
    window_minutes: list[int],
    feature_prefix: str,
) -> pd.DataFrame:
    """Attaches past event counts within rolling time windows.

    For target timestamp t and window w, events are counted when:

        t - w <= event time < t

    Events occurring exactly at the target timestamp are excluded.

    Args:
        target_table: Table receiving rolling count features.
        event_table: Table containing historical events.
        group_columns: Columns defining independent histories.
        target_time_column: Timestamp at which counts are measured.
        event_time_column: Timestamp when each event occurred.
        window_minutes: Rolling-window lengths in minutes.
        feature_prefix: Prefix used in generated feature names.

    Returns:
        A copy of the target table with rolling count features.
    """
    result_groups = []

    for group_key, target_group in target_table.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        event_mask = pd.Series(
            True,
            index=event_table.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= event_table[column].isna()
            else:
                event_mask &= event_table[column].eq(value)

        group_events = (
            event_table.loc[
                event_mask,
                event_time_column,
            ]
            .dropna()
            .sort_values()
        )

        current_group = (
            target_group
            .sort_values(target_time_column)
            .copy()
        )

        target_times = (
            current_group[target_time_column]
            .to_numpy(dtype="datetime64[ns]")
        )

        event_times = (
            group_events
            .to_numpy(dtype="datetime64[ns]")
        )

        for window in window_minutes:
            left_boundaries = (
                target_times
                - np.timedelta64(window, "m")
            )

            left_indices = np.searchsorted(
                event_times,
                left_boundaries,
                side="left",
            )

            right_indices = np.searchsorted(
                event_times,
                target_times,
                side="left",
            )

            feature_column = (
                f"{feature_prefix}_past_{window}_minutes"
            )

            current_group[feature_column] = (
                right_indices - left_indices
            )

        result_groups.append(current_group)

    return pd.concat(
        result_groups,
        ignore_index=True,
    )

### Calculate rolling trade-opening counts

In [ ]:
trade_features_table = attach_rolling_event_counts(
    target_table=trade_features_table,
    event_table=trades,
    group_columns=[
        "account_id",
        "campaign_id",
    ],
    target_time_column="open_date_time",
    event_time_column="open_date_time",
    window_minutes=[
        15,
        30,
        60,
    ],
    feature_prefix="trades_opened",
)

### Calculate rolling idea-start counts

In [ ]:
trade_features_table = attach_rolling_event_counts(
    target_table=trade_features_table,
    event_table=idea_start_events,
    group_columns=[
        "account_id",
        "campaign_id",
    ],
    target_time_column="open_date_time",
    event_time_column="idea_start_time",
    window_minutes=[
        15,
        30,
        60,
    ],
    feature_prefix="ideas_started",
)

### Inspect the features

In [ ]:
intensity_feature_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "idea_id",
    "open_date_time",
    "trades_opened_past_15_minutes",
    "trades_opened_past_30_minutes",
    "trades_opened_past_60_minutes",
    "ideas_started_past_15_minutes",
    "ideas_started_past_30_minutes",
    "ideas_started_past_60_minutes",
]

display(
    trade_features_table[
        intensity_feature_columns
    ].head(30)
)

,trade_row_id,account_id,campaign_id,idea_id,open_date_time,trades_opened_past_15_minutes,trades_opened_past_30_minutes,trades_opened_past_60_minutes,ideas_started_past_15_minutes,ideas_started_past_30_minutes,ideas_started_past_60_minutes
0,1,D#1589378,33,D#1589378_33_sell_1,2026-02-24 12:01:08+00:00,0,0,0,0,0,0
1,2,D#1589408,33,D#1589408_33_sell_1,2026-02-24 02:46:35+00:00,0,0,0,0,0,0
2,3,D#1589408,33,D#1589408_33_buy_1,2026-02-24 02:53:40+00:00,1,1,1,1,1,1
3,4,D#1589408,33,D#1589408_33_sell_2,2026-02-24 02:57:37+00:00,2,2,2,2,2,2
4,5,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:13:04+00:00,0,0,0,0,0,0
5,6,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:23:39+00:00,1,1,1,1,1,1
6,7,D#1589419,33,D#1589419_33_buy_1,2026-02-24 05:32:22+00:00,0,0,0,0,0,0
7,8,D#1589428,33,D#1589428_33_buy_1,2026-02-24 02:38:47+00:00,0,0,0,0,0,0
8,9,D#1589428,33,D#1589428_33_buy_2,2026-02-24 07:24:19+00:00,0,0,0,0,0,0
9,10,D#1589428,33,D#1589428_33_buy_3,2026-02-24 08:38:11+00:00,0,0,0,0,0,0


### Basic validation

In [ ]:
trade_window_order_valid = (
    (
        trade_features_table[
            "trades_opened_past_15_minutes"
        ]
        <= trade_features_table[
            "trades_opened_past_30_minutes"
        ]
    )
    & (
        trade_features_table[
            "trades_opened_past_30_minutes"
        ]
        <= trade_features_table[
            "trades_opened_past_60_minutes"
        ]
    )
).all()

idea_window_order_valid = (
    (
        trade_features_table[
            "ideas_started_past_15_minutes"
        ]
        <= trade_features_table[
            "ideas_started_past_30_minutes"
        ]
    )
    & (
        trade_features_table[
            "ideas_started_past_30_minutes"
        ]
        <= trade_features_table[
            "ideas_started_past_60_minutes"
        ]
    )
).all()

print(
    "Trade rolling windows valid:",
    trade_window_order_valid,
)

print(
    "Idea rolling windows valid:",
    idea_window_order_valid,
)

Trade rolling windows valid: True
Idea rolling windows valid: True


In [ ]:
display(
    trade_features_table[
        intensity_feature_columns[5:]
    ].describe()
)

,trades_opened_past_15_minutes,trades_opened_past_30_minutes,trades_opened_past_60_minutes,ideas_started_past_15_minutes,ideas_started_past_30_minutes,ideas_started_past_60_minutes
count,46520.000000,46520.000000,46520.000000,46520.000000,46520.000000,46520.000000
mean,1.206234,1.879923,2.690413,0.629987,1.035641,1.573474
std,2.155429,3.128761,4.058800,0.836797,1.226608,1.787677
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000
75%,2.000000,2.000000,3.000000,1.000000,2.000000,2.000000
max,28.000000,48.000000,62.000000,6.000000,8.000000,12.000000


### Calculate recent activity shares

In [ ]:
for window in [15, 30, 60]:
    trade_count_column = (
        f"trades_opened_past_{window}_minutes"
    )
    trade_share_column = (
        f"recent_trade_share_{window}_minutes"
    )

    idea_count_column = (
        f"ideas_started_past_{window}_minutes"
    )
    idea_share_column = (
        f"recent_idea_share_{window}_minutes"
    )

    trade_features_table[trade_share_column] = np.where(
        trade_features_table[
            "past_trade_open_count"
        ].gt(0),
        (
            trade_features_table[
                trade_count_column
            ]
            / trade_features_table[
                "past_trade_open_count"
            ]
        ),
        np.nan,
    )

    trade_features_table[idea_share_column] = np.where(
        trade_features_table[
            "past_idea_start_count"
        ].gt(0),
        (
            trade_features_table[
                idea_count_column
            ]
            / trade_features_table[
                "past_idea_start_count"
            ]
        ),
        np.nan,
    )

### Add recent activity share availability indicators

In [ ]:
trade_features_table[
    "recent_trade_share_available"
] = (
    trade_features_table[
        "past_trade_open_count"
    ].gt(0)
)

trade_features_table[
    "recent_idea_share_available"
] = (
    trade_features_table[
        "past_idea_start_count"
    ].gt(0)
)

### Inspect recent activity shares

In [ ]:
activity_share_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "idea_id",
    "open_date_time",
    "past_trade_open_count",
    "trades_opened_past_15_minutes",
    "trades_opened_past_30_minutes",
    "trades_opened_past_60_minutes",
    "recent_trade_share_15_minutes",
    "recent_trade_share_30_minutes",
    "recent_trade_share_60_minutes",
    "recent_trade_share_available",
    "past_idea_start_count",
    "ideas_started_past_15_minutes",
    "ideas_started_past_30_minutes",
    "ideas_started_past_60_minutes",
    "recent_idea_share_15_minutes",
    "recent_idea_share_30_minutes",
    "recent_idea_share_60_minutes",
    "recent_idea_share_available",
]

display(
    trade_features_table[
        activity_share_columns
    ].head(30)
)

,trade_row_id,account_id,campaign_id,idea_id,open_date_time,past_trade_open_count,trades_opened_past_15_minutes,trades_opened_past_30_minutes,trades_opened_past_60_minutes,recent_trade_share_15_minutes,...,recent_trade_share_60_minutes,recent_trade_share_available,past_idea_start_count,ideas_started_past_15_minutes,ideas_started_past_30_minutes,ideas_started_past_60_minutes,recent_idea_share_15_minutes,recent_idea_share_30_minutes,recent_idea_share_60_minutes,recent_idea_share_available
0,1,D#1589378,33,D#1589378_33_sell_1,2026-02-24 12:01:08+00:00,0,0,0,0,NaN,...,NaN,False,0,0,0,0,NaN,NaN,NaN,False
1,2,D#1589408,33,D#1589408_33_sell_1,2026-02-24 02:46:35+00:00,0,0,0,0,NaN,...,NaN,False,0,0,0,0,NaN,NaN,NaN,False
2,3,D#1589408,33,D#1589408_33_buy_1,2026-02-24 02:53:40+00:00,1,1,1,1,1.000000,...,1.000000,True,1,1,1,1,1.000000,1.000000,1.000000,True
3,4,D#1589408,33,D#1589408_33_sell_2,2026-02-24 02:57:37+00:00,2,2,2,2,1.000000,...,1.000000,True,2,2,2,2,1.000000,1.000000,1.000000,True
4,5,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:13:04+00:00,0,0,0,0,NaN,...,NaN,False,0,0,0,0,NaN,NaN,NaN,False
5,6,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:23:39+00:00,1,1,1,1,1.000000,...,1.000000,True,1,1,1,1,1.000000,1.000000,1.000000,True
6,7,D#1589419,33,D#1589419_33_buy_1,2026-02-24 05:32:22+00:00,0,0,0,0,NaN,...,NaN,False,0,0,0,0,NaN,NaN,NaN,False
7,8,D#1589428,33,D#1589428_33_buy_1,2026-02-24 02:38:47+00:00,0,0,0,0,NaN,...,NaN,False,0,0,0,0,NaN,NaN,NaN,False
8,9,D#1589428,33,D#1589428_33_buy_2,2026-02-24 07:24:19+00:00,1,0,0,0,0.000000,...,0.000000,True,1,0,0,0,0.000000,0.000000,0.000000,True
9,10,D#1589428,33,D#1589428_33_buy_3,2026-02-24 08:38:11+00:00,2,0,0,0,0.000000,...,0.000000,True,2,0,0,0,0.000000,0.000000,0.000000,True


### Basic validation

In [ ]:
trade_share_columns = [
    "recent_trade_share_15_minutes",
    "recent_trade_share_30_minutes",
    "recent_trade_share_60_minutes",
]

idea_share_columns = [
    "recent_idea_share_15_minutes",
    "recent_idea_share_30_minutes",
    "recent_idea_share_60_minutes",
]

trade_share_range_valid = (
    trade_features_table[
        trade_share_columns
    ]
    .stack()
    .between(0, 1)
    .all()
)

idea_share_range_valid = (
    trade_features_table[
        idea_share_columns
    ]
    .stack()
    .between(0, 1)
    .all()
)

print(
    "Trade activity shares are between 0 and 1:",
    trade_share_range_valid,
)

print(
    "Idea activity shares are between 0 and 1:",
    idea_share_range_valid,
)

Trade activity shares are between 0 and 1: False
Idea activity shares are between 0 and 1: False


In [ ]:
trade_share_order_valid = (
    (
        trade_features_table[
            "recent_trade_share_15_minutes"
        ]
        <= trade_features_table[
            "recent_trade_share_30_minutes"
        ]
    )
    & (
        trade_features_table[
            "recent_trade_share_30_minutes"
        ]
        <= trade_features_table[
            "recent_trade_share_60_minutes"
        ]
    )
).loc[
    trade_features_table[
        "recent_trade_share_available"
    ]
].all()

idea_share_order_valid = (
    (
        trade_features_table[
            "recent_idea_share_15_minutes"
        ]
        <= trade_features_table[
            "recent_idea_share_30_minutes"
        ]
    )
    & (
        trade_features_table[
            "recent_idea_share_30_minutes"
        ]
        <= trade_features_table[
            "recent_idea_share_60_minutes"
        ]
    )
).loc[
    trade_features_table[
        "recent_idea_share_available"
    ]
].all()

print(
    "Trade activity share windows valid:",
    trade_share_order_valid,
)

print(
    "Idea activity share windows valid:",
    idea_share_order_valid,
)

Trade activity share windows valid: True
Idea activity share windows valid: True


In [ ]:
trade_unavailable_values_missing = (
    trade_features_table.loc[
        ~trade_features_table[
            "recent_trade_share_available"
        ],
        trade_share_columns,
    ]
    .isna()
    .all()
    .all()
)

idea_unavailable_values_missing = (
    trade_features_table.loc[
        ~trade_features_table[
            "recent_idea_share_available"
        ],
        idea_share_columns,
    ]
    .isna()
    .all()
    .all()
)

print(
    "Unavailable trade shares are missing:",
    trade_unavailable_values_missing,
)

print(
    "Unavailable idea shares are missing:",
    idea_unavailable_values_missing,
)

Unavailable trade shares are missing: True
Unavailable idea shares are missing: True


### Inspect the distributions

In [ ]:
display(
    trade_features_table[
        trade_share_columns
        + idea_share_columns
    ].describe()
)

,recent_trade_share_15_minutes,recent_trade_share_30_minutes,recent_trade_share_60_minutes,recent_idea_share_15_minutes,recent_idea_share_30_minutes,recent_idea_share_60_minutes
count,38355.000000,38355.000000,38355.000000,38355.000000,38355.000000,38355.000000
mean,0.283125,0.405494,0.537738,0.262770,0.388183,0.524835
std,0.359438,0.393837,0.398373,0.355859,0.393213,0.400433
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.166667,0.000000,0.000000,0.142857
50%,0.111111,0.266667,0.500000,0.083333,0.250000,0.500000
75%,0.500000,0.833333,1.000000,0.400000,0.750000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### Walk-forward testing

In [ ]:
recent_intensity_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "trades_opened_past_30_minutes"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "trades_opened_past_30_minutes",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [ ]:
def learn_high_recent_intensity_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually high recent trading intensity."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "trades_opened_past_30_minutes"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "trades_opened_past_30_minutes"
            ]
            >= threshold
        )
        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": signal_mean - control_mean,
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result["training_uplift"],
    )

In [ ]:
def walk_forward_high_recent_intensity(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually high recent trading intensity."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = campaign_order[:test_index]
        test_campaign = campaign_order[test_index]

        train_data = data.loc[
            data["campaign_id"].isin(training_campaigns)
        ]

        test_data = data.loc[
            data["campaign_id"].eq(test_campaign)
        ].copy()

        threshold_result = (
            learn_high_recent_intensity_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result["threshold"]

        signal_mask = (
            test_data[
                "trades_opened_past_30_minutes"
            ]
            >= threshold
        )

        signal_data = test_data.loc[signal_mask]
        control_data = test_data.loc[~signal_mask]

        signal_mean = (
            signal_data["reverse_profit_per_lot"].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data["reverse_profit_per_lot"].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": threshold_result[
                    "quantile"
                ],
                "learned_threshold": threshold,
                "test_signal_trades": len(signal_data),
                "test_control_trades": len(control_data),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(fold_results)

In [97]:
recent_intensity_walk_forward = (
    walk_forward_high_recent_intensity(
        recent_intensity_test_data,
        minimum_training_campaigns=10,
    )
)

display(recent_intensity_walk_forward)

summarize_walk_forward(
    recent_intensity_walk_forward,
    "High recent trading intensity (30 min)",
)

NameError: name 'walk_forward_high_recent_intensity' is not defined

## Trading pace and spacing

### Define entry-spacing function

In [98]:
def attach_entry_spacing_features(
    target_table: pd.DataFrame,
    event_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    event_time_column: str,
    tie_breaker_columns: list[str],
    previous_gap_column: str,
    past_median_gap_column: str,
    gap_ratio_column: str,
    gap_available_column: str,
    median_available_column: str,
) -> pd.DataFrame:
    """Attaches previous and historical median entry-gap features.

    For every target row, only events occurring strictly before the
    target timestamp are used.

    Args:
        target_table: Table receiving the spacing features.
        event_table: Table containing trade-opening or idea-start events.
        group_columns: Columns defining independent trading histories.
        target_time_column: Timestamp at which features are measured.
        event_time_column: Timestamp when each historical event occurred.
        tie_breaker_columns: Columns providing deterministic ordering for
            events sharing the same timestamp.
        previous_gap_column: Output name for the time since the latest
            strictly earlier event.
        past_median_gap_column: Output name for the median gap among
            strictly earlier events.
        gap_ratio_column: Output name for the current gap divided by the
            historical median gap.
        gap_available_column: Output name indicating whether a previous
            event exists.
        median_available_column: Output name indicating whether a valid
            historical median exists.

    Returns:
        A copy of the target table with entry-spacing features attached.
    """
    target = target_table.copy()
    events = event_table.copy()

    output_groups = []

    for group_key, target_group in target.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        event_mask = pd.Series(
            True,
            index=events.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= events[column].isna()
            else:
                event_mask &= events[column].eq(value)

        group_events = (
            events.loc[event_mask]
            .sort_values(
                [event_time_column]
                + tie_breaker_columns
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(target_time_column)
            .copy()
        )

        if group_events.empty:
            current_group[previous_gap_column] = np.nan
            current_group[past_median_gap_column] = np.nan
            current_group[gap_ratio_column] = np.nan
            current_group[gap_available_column] = False
            current_group[median_available_column] = False

            output_groups.append(current_group)
            continue

        event_gap_column = "_entry_gap_minutes"

        group_events[event_gap_column] = (
            group_events[event_time_column]
            .diff()
            .dt.total_seconds()
            / 60
        )

        group_events[past_median_gap_column] = (
            group_events[event_gap_column]
            .expanding(min_periods=1)
            .median()
        )

        merge_time_column = "_previous_event_time"

        events_for_merge = (
            group_events[
                [
                    event_time_column,
                    past_median_gap_column,
                ]
            ]
            .rename(
                columns={
                    event_time_column: merge_time_column,
                }
            )
        )

        current_group = pd.merge_asof(
            left=current_group,
            right=events_for_merge,
            left_on=target_time_column,
            right_on=merge_time_column,
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[previous_gap_column] = (
            current_group[target_time_column]
            - current_group[merge_time_column]
        ).dt.total_seconds() / 60

        current_group[gap_available_column] = (
            current_group[merge_time_column].notna()
        )

        current_group[median_available_column] = (
            current_group[past_median_gap_column].gt(0)
        )

        valid_ratio = (
            current_group[previous_gap_column].ge(0)
            & current_group[past_median_gap_column].gt(0)
        )

        current_group[gap_ratio_column] = np.where(
            valid_ratio,
            (
                current_group[previous_gap_column]
                / current_group[past_median_gap_column]
            ),
            np.nan,
        )

        current_group = current_group.drop(
            columns=[merge_time_column],
            errors="ignore",
        )

        output_groups.append(current_group)

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

### Attach trade-level spacing features

In [99]:
trade_features_table = attach_entry_spacing_features(
    target_table=trade_features_table,
    event_table=trades,
    group_columns=[
        "account_id",
        "campaign_id",
    ],
    target_time_column="open_date_time",
    event_time_column="open_date_time",
    tie_breaker_columns=[
        "trade_row_id",
    ],
    previous_gap_column=(
        "minutes_since_previous_trade_open"
    ),
    past_median_gap_column=(
        "past_median_trade_open_gap_minutes"
    ),
    gap_ratio_column=(
        "trade_open_gap_to_past_median_ratio"
    ),
    gap_available_column=(
        "previous_trade_open_gap_available"
    ),
    median_available_column=(
        "past_median_trade_open_gap_available"
    ),
)

### Attach idea-level spacing features

In [100]:
idea_features_table = attach_entry_spacing_features(
    target_table=idea_features_table,
    event_table=idea_start_events,
    group_columns=[
        "account_id",
        "campaign_id",
    ],
    target_time_column="idea_start_time",
    event_time_column="idea_start_time",
    tie_breaker_columns=[
        "idea_id",
    ],
    previous_gap_column=(
        "minutes_since_previous_idea_start"
    ),
    past_median_gap_column=(
        "past_median_idea_start_gap_minutes"
    ),
    gap_ratio_column=(
        "idea_start_gap_to_past_median_ratio"
    ),
    gap_available_column=(
        "previous_idea_start_gap_available"
    ),
    median_available_column=(
        "past_median_idea_start_gap_available"
    ),
)

### Restore the standard row order

In [101]:
trade_features_table = (
    trade_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "trade_row_id",
        ]
    )
    .reset_index(drop=True)
)

idea_features_table = (
    idea_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "idea_start_time",
            "idea_end_time",
            "idea_id",
        ]
    )
    .reset_index(drop=True)
)

### Inspect trade-level features

In [102]:
trade_spacing_feature_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "idea_id",
    "open_date_time",
    "minutes_since_previous_trade_open",
    "past_median_trade_open_gap_minutes",
    "trade_open_gap_to_past_median_ratio",
    "previous_trade_open_gap_available",
    "past_median_trade_open_gap_available",
]

display(
    trade_features_table[
        trade_spacing_feature_columns
    ].head(30)
)

,trade_row_id,account_id,campaign_id,idea_id,open_date_time,minutes_since_previous_trade_open,past_median_trade_open_gap_minutes,trade_open_gap_to_past_median_ratio,previous_trade_open_gap_available,past_median_trade_open_gap_available
0,1,D#1589378,33,D#1589378_33_sell_1,2026-02-24 12:01:08+00:00,NaN,NaN,NaN,False,False
1,2,D#1589408,33,D#1589408_33_sell_1,2026-02-24 02:46:35+00:00,NaN,NaN,NaN,False,False
2,3,D#1589408,33,D#1589408_33_buy_1,2026-02-24 02:53:40+00:00,7.083333,NaN,NaN,True,False
3,4,D#1589408,33,D#1589408_33_sell_2,2026-02-24 02:57:37+00:00,3.950000,7.083333,0.557647,True,True
4,5,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:13:04+00:00,NaN,NaN,NaN,False,False
5,6,D#1589410,33,D#1589410_33_buy_1,2026-02-25 00:23:39+00:00,10.583333,NaN,NaN,True,False
6,7,D#1589419,33,D#1589419_33_buy_1,2026-02-24 05:32:22+00:00,NaN,NaN,NaN,False,False
7,8,D#1589428,33,D#1589428_33_buy_1,2026-02-24 02:38:47+00:00,NaN,NaN,NaN,False,False
8,9,D#1589428,33,D#1589428_33_buy_2,2026-02-24 07:24:19+00:00,285.533333,NaN,NaN,True,False
9,10,D#1589428,33,D#1589428_33_buy_3,2026-02-24 08:38:11+00:00,73.866667,285.533333,0.258697,True,True


### Inspect idea-level features

In [103]:
idea_spacing_feature_columns = [
    "idea_id",
    "account_id",
    "campaign_id",
    "idea_start_time",
    "minutes_since_previous_idea_start",
    "past_median_idea_start_gap_minutes",
    "idea_start_gap_to_past_median_ratio",
    "previous_idea_start_gap_available",
    "past_median_idea_start_gap_available",
]

display(
    idea_features_table[
        idea_spacing_feature_columns
    ].head(30)
)

,idea_id,account_id,campaign_id,idea_start_time,minutes_since_previous_idea_start,past_median_idea_start_gap_minutes,idea_start_gap_to_past_median_ratio,previous_idea_start_gap_available,past_median_idea_start_gap_available
0,D#1589378_33_sell_1,D#1589378,33,2026-02-24 12:01:08+00:00,NaN,NaN,NaN,False,False
1,D#1589408_33_sell_1,D#1589408,33,2026-02-24 02:46:35+00:00,NaN,NaN,NaN,False,False
2,D#1589408_33_buy_1,D#1589408,33,2026-02-24 02:53:40+00:00,7.083333,NaN,NaN,True,False
3,D#1589408_33_sell_2,D#1589408,33,2026-02-24 02:57:37+00:00,3.950000,7.083333,0.557647,True,True
4,D#1589410_33_buy_1,D#1589410,33,2026-02-25 00:13:04+00:00,NaN,NaN,NaN,False,False
5,D#1589419_33_buy_1,D#1589419,33,2026-02-24 05:32:22+00:00,NaN,NaN,NaN,False,False
6,D#1589428_33_buy_1,D#1589428,33,2026-02-24 02:38:47+00:00,NaN,NaN,NaN,False,False
7,D#1589428_33_buy_2,D#1589428,33,2026-02-24 07:24:19+00:00,285.533333,NaN,NaN,True,False
8,D#1589428_33_buy_3,D#1589428,33,2026-02-24 08:38:11+00:00,73.866667,285.533333,0.258697,True,True
9,D#1614600_33_buy_1,D#1614600,33,2026-02-24 11:26:56+00:00,NaN,NaN,NaN,False,False


### Validation

In [104]:
trade_gap_nonnegative = (
    trade_features_table[
        "minutes_since_previous_trade_open"
    ]
    .dropna()
    .ge(0)
    .all()
)

idea_gap_nonnegative = (
    idea_features_table[
        "minutes_since_previous_idea_start"
    ]
    .dropna()
    .ge(0)
    .all()
)

print(
    "Trade-opening gaps are non-negative:",
    trade_gap_nonnegative,
)

print(
    "Idea-start gaps are non-negative:",
    idea_gap_nonnegative,
)

Trade-opening gaps are non-negative: True
Idea-start gaps are non-negative: True


In [105]:
print(
    trade_features_table[
        "previous_trade_open_gap_available"
    ].value_counts(dropna=False)
)

print(
    idea_features_table[
        "previous_idea_start_gap_available"
    ].value_counts(dropna=False)
)

previous_trade_open_gap_available
True     38355
False     8165
Name: count, dtype: int64
previous_idea_start_gap_available
True     25214
False     8165
Name: count, dtype: int64


### Inspect distribution

In [106]:
display(
    trade_features_table[
        [
            "minutes_since_previous_trade_open",
            "past_median_trade_open_gap_minutes",
            "trade_open_gap_to_past_median_ratio",
        ]
    ].describe()
)

display(
    idea_features_table[
        [
            "minutes_since_previous_idea_start",
            "past_median_idea_start_gap_minutes",
            "idea_start_gap_to_past_median_ratio",
        ]
    ].describe()
)

,minutes_since_previous_trade_open,past_median_trade_open_gap_minutes,trade_open_gap_to_past_median_ratio
count,38355.000000,31957.000000,31957.000000
mean,43.906374,31.482645,4.787180
std,94.108950,62.753131,39.389092
min,0.050000,0.083333,0.000367
25%,2.916667,4.266667,0.307810
50%,9.883333,11.066667,0.826196
75%,36.533333,29.450000,2.168224
max,1382.066667,1382.066667,2840.833333


,minutes_since_previous_idea_start,past_median_idea_start_gap_minutes,idea_start_gap_to_past_median_ratio
count,25214.000000,19436.000000,19436.000000
mean,65.089943,50.694918,3.337453
std,111.106565,78.862287,25.807830
min,0.083333,0.100000,0.000679
25%,8.950000,11.783333,0.357550
50%,22.116667,23.283333,0.869799
75%,67.400000,53.537500,2.098683
max,1382.066667,1382.066667,1797.187500


### Walk-forward testing

In [107]:
trading_pace_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "trade_open_gap_to_past_median_ratio"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "trade_open_gap_to_past_median_ratio",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [108]:
def learn_low_trading_pace_ratio_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually fast trading pace."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "trade_open_gap_to_past_median_ratio"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "trade_open_gap_to_past_median_ratio"
            ]
            <= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result["training_uplift"],
    )

In [109]:
def walk_forward_fast_trading_pace(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually fast trading pace."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = campaign_order[:test_index]
        test_campaign = campaign_order[test_index]

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_low_trading_pace_ratio_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result["threshold"]

        signal_mask = (
            test_data[
                "trade_open_gap_to_past_median_ratio"
            ]
            <= threshold
        )

        signal_data = test_data.loc[signal_mask]
        control_data = test_data.loc[~signal_mask]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result["quantile"]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": len(
                    signal_data
                ),
                "test_control_trades": len(
                    control_data
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(fold_results)

In [110]:
trading_pace_walk_forward = (
    walk_forward_fast_trading_pace(
        trading_pace_test_data,
        minimum_training_campaigns=10,
    )
)

display(trading_pace_walk_forward)

summarize_walk_forward(
    trading_pace_walk_forward,
    "Unusually fast trading pace",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.5,0.816805,482,516,3.727692
1,44,0.3,0.389621,308,798,18.697084
2,45,0.3,0.392940,262,584,47.786558
3,46,0.3,0.392079,295,614,-75.526848
4,47,0.4,0.577247,238,386,-41.586611
5,48,0.5,0.827742,516,497,-46.125230
6,49,0.5,0.825581,434,443,44.578420
7,50,0.5,0.826415,381,353,-50.791658
8,51,0.5,0.824256,393,388,35.595465
9,52,0.2,0.235053,142,505,90.277913


Unusually fast trading pace
Mean OOS uplift: 4.735203640398214
Median OOS uplift: 18.091763132069254
Positive campaign share: 0.625
95% bootstrap CI: (np.float64(-15.478032489650596), np.float64(23.810290123468675))


## Concurrent exposure

### Define concurrent-exposure function

In [111]:
def attach_concurrent_exposure_features(
    target_table: pd.DataFrame,
    event_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    event_start_time_column: str,
    event_end_time_column: str,
    active_count_column: str,
    amount_column: str | None = None,
    active_amount_column: str | None = None,
) -> pd.DataFrame:
    """Attaches exposure that was active before each target event.

    An event is treated as active at the target timestamp when its
    start time is strictly earlier than the target timestamp and its
    end time is strictly later than the target timestamp.

    Events starting at exactly the target timestamp are excluded, so
    simultaneous entries do not count one another.

    Args:
        target_table: Table receiving the concurrent-exposure features.
        event_table: Table containing events that may remain active.
        group_columns: Columns defining independent trading histories.
        target_time_column: Timestamp when exposure is measured.
        event_start_time_column: Timestamp when each event started.
        event_end_time_column: Timestamp when each event ended.
        active_count_column: Output name for the number of active events.
        amount_column: Optional event-size column.
        active_amount_column: Optional output name for total active size.

    Returns:
        A copy of the target table with concurrent-exposure features.
    """
    target = target_table.copy()
    events = event_table.copy()

    output_groups = []

    for group_key, target_group in target.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        event_mask = pd.Series(
            True,
            index=events.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= events[column].isna()
            else:
                event_mask &= events[column].eq(value)

        group_events = (
            events.loc[event_mask]
            .dropna(
                subset=[
                    event_start_time_column,
                    event_end_time_column,
                ]
            )
            .copy()
        )

        current_group = target_group.copy()

        target_times = current_group[
            target_time_column
        ].to_numpy(
            dtype="datetime64[ns]"
        )

        if group_events.empty:
            current_group[active_count_column] = 0

            if active_amount_column is not None:
                current_group[active_amount_column] = 0.0

            output_groups.append(current_group)
            continue

        events_by_start = group_events.sort_values(
            event_start_time_column
        )

        events_by_end = group_events.sort_values(
            event_end_time_column
        )

        start_times = events_by_start[
            event_start_time_column
        ].to_numpy(
            dtype="datetime64[ns]"
        )

        end_times = events_by_end[
            event_end_time_column
        ].to_numpy(
            dtype="datetime64[ns]"
        )

        number_started_before = np.searchsorted(
            start_times,
            target_times,
            side="left",
        )

        number_ended_by_entry = np.searchsorted(
            end_times,
            target_times,
            side="right",
        )

        active_counts = (
            number_started_before
            - number_ended_by_entry
        )

        current_group[active_count_column] = np.maximum(
            active_counts,
            0,
        )

        if (
            amount_column is not None
            and active_amount_column is not None
        ):
            start_amounts = (
                events_by_start[amount_column]
                .fillna(0)
                .to_numpy(dtype=float)
            )

            end_amounts = (
                events_by_end[amount_column]
                .fillna(0)
                .to_numpy(dtype=float)
            )

            cumulative_start_amount = np.concatenate(
                [
                    [0.0],
                    np.cumsum(start_amounts),
                ]
            )

            cumulative_end_amount = np.concatenate(
                [
                    [0.0],
                    np.cumsum(end_amounts),
                ]
            )

            active_amounts = (
                cumulative_start_amount[
                    number_started_before
                ]
                - cumulative_end_amount[
                    number_ended_by_entry
                ]
            )

            current_group[active_amount_column] = np.maximum(
                active_amounts,
                0.0,
            )

        output_groups.append(current_group)

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

### Calculate active trade exposure

In [112]:
trade_features_table = (
    attach_concurrent_exposure_features(
        target_table=trade_features_table,
        event_table=trades,
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column="open_date_time",
        event_start_time_column="open_date_time",
        event_end_time_column="close_date_time",
        active_count_column=(
            "active_trade_count_before_entry"
        ),
        amount_column="amount",
        active_amount_column=(
            "active_amount_before_entry"
        ),
    )
)

In [113]:
trade_features_table[
    "active_amount_to_current_amount_ratio"
] = np.where(
    trade_features_table["amount"].gt(0),
    (
        trade_features_table[
            "active_amount_before_entry"
        ]
        / trade_features_table["amount"]
    ),
    np.nan,
)

In [114]:
trade_features_table[
    "has_concurrent_exposure"
] = (
    trade_features_table[
        "active_trade_count_before_entry"
    ] > 0
)

trade_features_table[
    "active_amount_to_current_amount_ratio"
] = np.where(
    trade_features_table["amount"] > 0,
    (
        trade_features_table[
            "active_amount_before_entry"
        ]
        / trade_features_table["amount"]
    ),
    np.nan,
)

In [115]:
assert len(trade_features_table) == len(trades)

assert (
    trade_features_table["trade_row_id"].nunique()
    == len(trade_features_table)
)

In [116]:
display(
    trade_features_table[
        [
            "active_trade_count_before_entry",
            "active_amount_before_entry",
            "active_amount_to_current_amount_ratio",
        ]
    ].describe()
)

,active_trade_count_before_entry,active_amount_before_entry,active_amount_to_current_amount_ratio
count,46520.000000,46520.000000,46520.000000
mean,0.000043,0.000002,0.000086
std,0.006557,0.000270,0.013517
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,1.000000,0.050000,2.500000


In [117]:
display(
    trade_features_table[
        "active_trade_count_before_entry"
    ]
    .value_counts()
    .sort_index()
    .head(20)
)

active_trade_count_before_entry
0    46518
1        2
Name: count, dtype: int64

In [118]:
print(
    "Trades with concurrent exposure:",
    trade_features_table[
        "has_concurrent_exposure"
    ].mean()
)

Trades with concurrent exposure: 4.299226139294927e-05


# Challenge-state / drawdown pressure

## Configuration

In [119]:
STARTING_BALANCE = 5_000.0
LOSS_LIMIT_RATE = 0.04
PROFIT_TARGET_RATE = 0.08

REALIZED_LOSS_BOUNDARY = (
    -STARTING_BALANCE
    * LOSS_LIMIT_RATE
)

PROFIT_TARGET_AMOUNT = (
    STARTING_BALANCE
    * PROFIT_TARGET_RATE
)

print(
    "Realized loss boundary:",
    REALIZED_LOSS_BOUNDARY,
)

print(
    "Profit target:",
    PROFIT_TARGET_AMOUNT,
)

Realized loss boundary: -200.0
Profit target: 400.0


## Build cumulative realized P&L events

In [120]:
realized_pnl_events = (
    trades[
        [
            "account_id",
            "campaign_id",
            "close_date_time",
            "net_profit",
        ]
    ]
    .dropna(
        subset=[
            "close_date_time",
            "net_profit",
        ]
    )
    .groupby(
        [
            "account_id",
            "campaign_id",
            "close_date_time",
        ],
        as_index=False,
    )
    .agg(
        realized_pnl_at_close=(
            "net_profit",
            "sum",
        )
    )
)

In [121]:
realized_pnl_events = (
    realized_pnl_events
    .sort_values(
        [
            "account_id",
            "campaign_id",
            "close_date_time",
        ]
    )
    .reset_index(drop=True)
)

realized_pnl_events[
    "cumulative_realized_pnl"
] = (
    realized_pnl_events
    .groupby(
        [
            "account_id",
            "campaign_id",
        ]
    )[
        "realized_pnl_at_close"
    ]
    .cumsum()
)

## Attach the latest completed realized state to each trade

In [122]:
def attach_realized_challenge_state(
    target_table: pd.DataFrame,
    realized_events: pd.DataFrame,
) -> pd.DataFrame:
    """Attaches cumulative realized P&L known at each trade opening.

    For each account-campaign, the most recent cumulative realized
    P&L state whose close timestamp is on or before the current
    trade's open timestamp is attached.

    Args:
        target_table: One-row-per-trade feature table.
        realized_events: Realized-P&L events containing cumulative
            realized P&L by account and campaign.

    Returns:
        Feature table with realized challenge-state information.
    """
    result_frames = []

    group_columns = [
        "campaign_id",
        "account_id",
    ]

    for group_key, target_group in target_table.groupby(
        group_columns,
        dropna=False,
        sort=False,
    ):
        campaign_id, account_id = group_key

        current = (
            target_group
            .sort_values(
                [
                    "open_date_time",
                    "close_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        event_mask = (
            realized_events["campaign_id"].eq(
                campaign_id
            )
            & realized_events["account_id"].eq(
                account_id
            )
        )

        events = (
            realized_events.loc[
                event_mask,
                [
                    "close_date_time",
                    "cumulative_realized_pnl",
                ],
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        if events.empty:
            current[
                "latest_realized_close_before_trade"
            ] = pd.NaT

            current[
                "realized_pnl_before_trade"
            ] = 0.0

            result_frames.append(current)
            continue

        events = events.rename(
            columns={
                "close_date_time": (
                    "latest_realized_close_before_trade"
                ),
                "cumulative_realized_pnl": (
                    "realized_pnl_before_trade"
                ),
            }
        )

        merged = pd.merge_asof(
            current.sort_values(
                "open_date_time"
            ),
            events.sort_values(
                "latest_realized_close_before_trade"
            ),
            left_on="open_date_time",
            right_on=(
                "latest_realized_close_before_trade"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        merged[
            "realized_pnl_before_trade"
        ] = (
            merged[
                "realized_pnl_before_trade"
            ]
            .fillna(0.0)
        )

        result_frames.append(merged)

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(drop=True)
    )

In [123]:
trade_features_table = (
    attach_realized_challenge_state(
        target_table=trade_features_table,
        realized_events=realized_pnl_events,
    )
)

## Construct the challenge-state features

In [124]:
trade_features_table[
    "realized_return_before_trade"
] = (
    trade_features_table[
        "realized_pnl_before_trade"
    ]
    / STARTING_BALANCE
)

In [125]:
trade_features_table[
    "realized_balance_before_trade"
] = (
    STARTING_BALANCE
    + trade_features_table[
        "realized_pnl_before_trade"
    ]
)

In [126]:
trade_features_table[
    "realized_distance_to_loss_limit"
] = (
    trade_features_table[
        "realized_pnl_before_trade"
    ]
    - REALIZED_LOSS_BOUNDARY
)

In [127]:
trade_features_table[
    "realized_distance_to_profit_target"
] = (
    PROFIT_TARGET_AMOUNT
    - trade_features_table[
        "realized_pnl_before_trade"
    ]
)

In [128]:
trade_features_table[
    "realized_distance_to_loss_limit_pct"
] = (
    trade_features_table[
        "realized_distance_to_loss_limit"
    ]
    / STARTING_BALANCE
)

trade_features_table[
    "realized_distance_to_profit_target_pct"
] = (
    trade_features_table[
        "realized_distance_to_profit_target"
    ]
    / STARTING_BALANCE
)

## Validate leakage

In [129]:
realized_state_time_valid = (
    trade_features_table[
        "latest_realized_close_before_trade"
    ].isna()
    | (
        trade_features_table[
            "latest_realized_close_before_trade"
        ]
        <= trade_features_table[
            "open_date_time"
        ]
    )
)

print(
    "Realized state uses only completed trades:",
    realized_state_time_valid.all(),
)

Realized state uses only completed trades: True


In [130]:
first_trade_mask = (
    trade_features_table[
        "open_date_time"
    ]
    == (
        trade_features_table
        .groupby(
            [
                "account_id",
                "campaign_id",
            ]
        )[
            "open_date_time"
        ]
        .transform("min")
    )
)

print(
    "First-entry realized P&L values:",
    trade_features_table.loc[
        first_trade_mask,
        "realized_pnl_before_trade",
    ]
    .value_counts()
    .head(10)
)

First-entry realized P&L values: realized_pnl_before_trade
 0.00    8162
-2.62       1
-7.80       1
-8.06       1
Name: count, dtype: int64


## Inspect rows

In [131]:
challenge_state_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "open_date_time",
    "latest_realized_close_before_trade",
    "realized_pnl_before_trade",
    "realized_balance_before_trade",
    "realized_return_before_trade",
    "realized_distance_to_loss_limit",
    "realized_distance_to_profit_target",
    "reverse_profit_per_lot",
]

display(
    trade_features_table[
        challenge_state_columns
    ].head(30)
)

,trade_row_id,account_id,campaign_id,open_date_time,latest_realized_close_before_trade,realized_pnl_before_trade,realized_balance_before_trade,realized_return_before_trade,realized_distance_to_loss_limit,realized_distance_to_profit_target,reverse_profit_per_lot
0,1,D#1589378,33,2026-02-24 12:01:08+00:00,NaT,0.00,5000.00,0.000000,200.00,400.00,-44.0
1,2,D#1589408,33,2026-02-24 02:46:35+00:00,NaT,0.00,5000.00,0.000000,200.00,400.00,395.0
2,3,D#1589408,33,2026-02-24 02:53:40+00:00,2026-02-24 02:52:47+00:00,-187.92,4812.08,-0.037584,12.08,587.92,-95.0
3,4,D#1589408,33,2026-02-24 02:57:37+00:00,2026-02-24 02:56:49+00:00,-182.62,4817.38,-0.036524,17.38,582.62,198.0
4,5,D#1589410,33,2026-02-25 00:13:04+00:00,NaT,0.00,5000.00,0.000000,200.00,400.00,-496.0
5,6,D#1589410,33,2026-02-25 00:23:39+00:00,2026-02-25 00:21:26+00:00,45.40,5045.40,0.009080,245.40,354.60,-464.0
6,7,D#1589419,33,2026-02-24 05:32:22+00:00,NaT,0.00,5000.00,0.000000,200.00,400.00,1968.0
7,8,D#1589428,33,2026-02-24 02:38:47+00:00,NaT,0.00,5000.00,0.000000,200.00,400.00,-172.0
8,9,D#1589428,33,2026-02-24 07:24:19+00:00,2026-02-24 02:44:09+00:00,13.00,5013.00,0.002600,213.00,387.00,-764.0
9,10,D#1589428,33,2026-02-24 08:38:11+00:00,2026-02-24 07:29:48+00:00,121.29,5121.29,0.024258,321.29,278.71,2187.0


In [132]:
display(
    trade_features_table[
        challenge_state_columns
    ]
    .sort_values(
        "realized_distance_to_loss_limit"
    )
    .head(30)
)

,trade_row_id,account_id,campaign_id,open_date_time,latest_realized_close_before_trade,realized_pnl_before_trade,realized_balance_before_trade,realized_return_before_trade,realized_distance_to_loss_limit,realized_distance_to_profit_target,reverse_profit_per_lot
893,894,D#1670956,34,2026-03-03 05:54:08+00:00,2026-03-03 05:54:01+00:00,-504.04,4495.96,-0.100808,-304.04,904.04,58.0
40154,40155,D#1702421,63,2026-06-23 08:03:45+00:00,2026-06-23 08:03:43+00:00,-356.47,4643.53,-0.071294,-156.47,756.47,10.0
25270,25271,D#1645696,53,2026-05-19 13:31:30+00:00,2026-05-19 13:31:27+00:00,-348.72,4651.28,-0.069744,-148.72,748.72,574.0
13272,13273,D#1702540,43,2026-04-06 07:06:05+00:00,2026-04-06 07:06:02+00:00,-310.60,4689.40,-0.062120,-110.60,710.60,69.0
735,736,D#1702737,33,2026-02-24 14:20:08+00:00,2026-02-24 13:59:31+00:00,-292.10,4707.90,-0.058420,-92.10,692.10,-2471.0
15718,15719,D#1670978,45,2026-04-14 04:02:35+00:00,2026-04-14 03:59:39+00:00,-289.09,4710.91,-0.057818,-89.09,689.09,486.0
14644,14645,D#1702462,44,2026-04-10 12:30:11+00:00,2026-04-10 12:30:06+00:00,-284.16,4715.84,-0.056832,-84.16,684.16,109.0
19909,19910,D#1702519,48,2026-04-29 00:01:13+00:00,2026-04-29 00:01:07+00:00,-281.65,4718.35,-0.056330,-81.65,681.65,102.0
19862,19863,D#1702481,48,2026-04-28 08:16:43+00:00,2026-04-28 08:11:51+00:00,-280.01,4719.99,-0.056002,-80.01,680.01,246.0
8079,8080,D#1589455,40,2026-03-24 04:28:38+00:00,2026-03-24 04:28:36+00:00,-265.80,4734.20,-0.053160,-65.80,665.80,1.0


In [133]:
display(
    trade_features_table[
        [
            "realized_pnl_before_trade",
            "realized_return_before_trade",
            "realized_distance_to_loss_limit",
            "realized_distance_to_profit_target",
        ]
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

,realized_pnl_before_trade,realized_return_before_trade,realized_distance_to_loss_limit,realized_distance_to_profit_target
count,46520.000000,46520.000000,46520.000000,46520.000000
mean,45.995902,0.009199,245.995902,354.004098
std,121.826696,0.024365,121.826696,121.826696
min,-504.040000,-0.100808,-304.040000,-41.000000
1%,-185.690500,-0.037138,14.309500,21.415200
5%,-131.500000,-0.026300,68.500000,101.600000
10%,-90.770000,-0.018154,109.230000,170.584000
25%,-14.325000,-0.002865,185.675000,291.377500
50%,3.385000,0.000677,203.385000,396.615000
75%,108.622500,0.021724,308.622500,414.325000


## Walk-forward testing

### Prepare realized distance to loss limit test data

In [134]:
drawdown_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "reverse_profit_per_lot"
        ].notna()
        & trade_features_table[
            "realized_distance_to_loss_limit"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "realized_pnl_before_trade",
            "realized_distance_to_loss_limit",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

### Learn drawdown pressure threshold

In [135]:
def learn_drawdown_pressure_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a drawdown-pressure threshold using training data only.

    Args:
        train_data: Historical campaign observations.
        candidate_quantiles: Candidate lower-tail quantiles for
            realized distance to the loss limit.
        min_signal_trades: Minimum signal observations required.
        min_control_trades: Minimum control observations required.

    Returns:
        Best threshold information, or None if no valid threshold exists.
    """
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "realized_distance_to_loss_limit"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "realized_distance_to_loss_limit"
            ]
            <= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_signal_mean": signal_mean,
                "training_control_mean": control_mean,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

### Walk-forward test function

In [136]:
def walk_forward_drawdown_pressure(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests realized drawdown pressure.

    The near-loss-limit threshold is learned exclusively from earlier
    campaigns and then frozen for the next campaign.

    Args:
        data: Trade-level challenge-state observations.
        minimum_training_campaigns: Number of historical campaigns
            required before testing.

    Returns:
        One row per OOS test campaign.
    """
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_drawdown_pressure_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "realized_distance_to_loss_limit"
            ]
            <= threshold
        )

        control_mask = ~signal_mask

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            control_mask
        ]

        # Keep the fold even when one group is absent.
        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "training_campaign_count": (
                    len(training_campaigns)
                ),
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": (
                    threshold
                ),
                "training_uplift": (
                    threshold_result[
                        "training_uplift"
                    ]
                ),
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [137]:
drawdown_walk_forward = (
    walk_forward_drawdown_pressure(
        drawdown_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    drawdown_walk_forward
)

,test_campaign,training_campaign_count,learned_quantile,learned_threshold,training_uplift,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,test_uplift_vs_control
0,43,10,0.3,200.0,28.265081,592,812,-44.543271,-90.604444,46.061172
1,44,11,0.3,200.0,30.899232,741,836,54.447368,-43.850558,98.297927
2,45,12,0.3,200.0,37.903034,546,709,3.748615,34.296783,-30.548168
3,46,13,0.3,200.0,32.647233,698,692,-51.731638,-33.723988,-18.007650
4,47,14,0.3,200.0,28.095603,505,481,-16.356880,0.086902,-16.443782
5,48,15,0.3,200.0,25.723262,682,787,21.110997,-30.188056,51.299053
6,49,16,0.3,200.0,27.560623,675,692,-87.804630,-30.552612,-57.252017
7,50,17,0.3,200.0,21.770163,633,538,63.772407,21.974071,41.798336
8,51,18,0.3,200.0,23.597130,613,552,26.637124,-41.642029,68.279153
9,52,19,0.3,200.0,25.746349,477,510,-14.376101,-4.150168,-10.225933


### Summarize OOS result

In [138]:
valid_drawdown_folds = (
    drawdown_walk_forward[
        "test_uplift_vs_control"
    ]
    .dropna()
)

print(
    "Mean OOS uplift:",
    valid_drawdown_folds.mean(),
)

print(
    "Median OOS uplift:",
    valid_drawdown_folds.median(),
)

print(
    "Positive-uplift campaign share:",
    (
        valid_drawdown_folds > 0
    ).mean(),
)

print(
    "Valid OOS campaigns:",
    len(valid_drawdown_folds),
)

print(
    "Total OOS campaigns:",
    len(drawdown_walk_forward),
)

Mean OOS uplift: 20.142013675879692
Median OOS uplift: 28.40602289786623
Positive-uplift campaign share: 0.5833333333333334
Valid OOS campaigns: 24
Total OOS campaigns: 24


In [139]:
drawdown_ci_lower, drawdown_ci_upper = (
    bootstrap_campaign_uplift_ci(
        drawdown_walk_forward
    )
)

print(
    "95% bootstrap CI:",
    (
        drawdown_ci_lower,
        drawdown_ci_upper,
    ),
)

95% bootstrap CI: (np.float64(3.3127180025343557), np.float64(37.32866762634906))


### Prepare profit-target test data

In [140]:
profit_target_test_data = (
    trade_features_table.loc[
        (
            trade_features_table[
                "realized_pnl_before_trade"
            ] > 0
        )
        & (
            trade_features_table[
                "realized_pnl_before_trade"
            ] < PROFIT_TARGET_AMOUNT
        )
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "realized_pnl_before_trade",
            "realized_distance_to_profit_target",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

print(
    "Eligible profit-state trades:",
    len(profit_target_test_data),
)

display(
    profit_target_test_data[
        [
            "realized_pnl_before_trade",
            "realized_distance_to_profit_target",
        ]
    ].describe()
)

Eligible profit-state trades: 24177


,realized_pnl_before_trade,realized_distance_to_profit_target
count,2.417700e+04,24177.000000
mean,1.296409e+02,270.359093
std,1.049849e+02,104.984858
min,1.421085e-14,0.020000
25%,4.185000e+01,199.210000
50%,1.034000e+02,296.600000
75%,2.007900e+02,358.150000
max,3.999800e+02,400.000000


### Learn threshold from earlier campaigns

In [141]:
def learn_profit_target_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a profit-target proximity threshold.

    Smaller distance means the account is closer to the profit target.

    Args:
        train_data: Historical trade observations.
        candidate_quantiles: Candidate lower-tail quantiles.
        min_signal_trades: Minimum signal trades required.
        min_control_trades: Minimum control trades required.

    Returns:
        Best training threshold information, or None.
    """
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "realized_distance_to_profit_target"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "realized_distance_to_profit_target"
            ]
            <= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_signal_mean": signal_mean,
                "training_control_mean": control_mean,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

### Walk-forward test

In [142]:
def walk_forward_profit_target_proximity(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests proximity to the realized profit target.

    The threshold is learned using earlier campaigns only and then
    frozen for the next OOS campaign.

    Args:
        data: Eligible profitable-state trade observations.
        minimum_training_campaigns: Minimum historical campaigns.

    Returns:
        One row per OOS campaign.
    """
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_profit_target_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "realized_distance_to_profit_target"
            ]
            <= threshold
        )

        control_mask = ~signal_mask

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            control_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "training_campaign_count": (
                    len(training_campaigns)
                ),
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "training_uplift": (
                    threshold_result[
                        "training_uplift"
                    ]
                ),
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [143]:
profit_target_walk_forward = (
    walk_forward_profit_target_proximity(
        profit_target_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    profit_target_walk_forward
)

,test_campaign,training_campaign_count,learned_quantile,learned_threshold,training_uplift,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,test_uplift_vs_control
0,43,10,0.1,100.990,-6.245226,71,741,-131.286385,-86.706444,-44.579941
1,44,11,0.3,217.951,-10.511066,208,628,-62.562500,-37.652972,-24.909528
2,45,12,0.1,106.710,-6.668848,51,658,153.725490,25.040151,128.685340
3,46,13,0.2,171.836,0.230285,129,563,117.976744,-68.483126,186.459870
4,47,14,0.2,172.320,12.626039,97,380,84.494845,-21.074211,105.569056
5,48,15,0.2,172.200,17.095966,163,624,-31.975460,-29.721154,-2.254306
6,49,16,0.2,172.024,15.925835,116,575,-47.810345,-27.075492,-20.734853
7,50,17,0.2,172.996,11.679170,96,442,13.778646,23.754072,-9.975427
8,51,18,0.2,173.602,10.584063,127,425,-91.425197,-26.765647,-64.659550
9,52,19,0.2,172.608,7.518392,113,397,23.143363,-11.918856,35.062219


### Summarize OOS performance

In [144]:
valid_profit_target_folds = (
    profit_target_walk_forward[
        "test_uplift_vs_control"
    ]
    .dropna()
)

print(
    "Mean OOS uplift:",
    valid_profit_target_folds.mean(),
)

print(
    "Median OOS uplift:",
    valid_profit_target_folds.median(),
)

print(
    "Positive-uplift campaign share:",
    (
        valid_profit_target_folds > 0
    ).mean(),
)

print(
    "Valid OOS campaigns:",
    len(valid_profit_target_folds),
)

print(
    "Total OOS campaigns:",
    len(profit_target_walk_forward),
)

Mean OOS uplift: 13.16112705548123
Median OOS uplift: -6.114866420701124
Positive-uplift campaign share: 0.4166666666666667
Valid OOS campaigns: 24
Total OOS campaigns: 24


In [145]:
profit_target_ci_lower, profit_target_ci_upper = (
    bootstrap_campaign_uplift_ci(
        profit_target_walk_forward
    )
)

print(
    "95% bootstrap CI:",
    (
        profit_target_ci_lower,
        profit_target_ci_upper,
    ),
)

95% bootstrap CI: (np.float64(-19.371522235770662), np.float64(48.747271025656595))


In [146]:
display(
    profit_target_walk_forward[
        [
            "test_campaign",
            "learned_quantile",
            "learned_threshold",
            "test_signal_trades",
            "test_control_trades",
            "test_uplift_vs_control",
        ]
    ]
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.1,100.990,71,741,-44.579941
1,44,0.3,217.951,208,628,-24.909528
2,45,0.1,106.710,51,658,128.685340
3,46,0.2,171.836,129,563,186.459870
4,47,0.2,172.320,97,380,105.569056
5,48,0.2,172.200,163,624,-2.254306
6,49,0.2,172.024,116,575,-20.734853
7,50,0.2,172.996,96,442,-9.975427
8,51,0.2,173.602,127,425,-64.659550
9,52,0.2,172.608,113,397,35.062219


## Drawdown + large size

In [147]:
def attach_historical_position_size_features(
    target_table: pd.DataFrame,
) -> pd.DataFrame:
    """Adds historical position-size features known before each trade.

    Historical size is based only on trades opened before the current
    trade within the same account and campaign.

    Args:
        target_table: One-row-per-trade feature table.

    Returns:
        Feature table with historical median size and current-to-history
        size ratio.
    """
    result_frames = []

    for _, group in target_table.groupby(
        [
            "campaign_id",
            "account_id",
        ],
        sort=False,
        dropna=False,
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_medians = []

        historical_amounts = []

        previous_open_time = None
        same_timestamp_amounts = []

        for row in current.itertuples():
            current_open_time = row.open_date_time

            if (
                previous_open_time is not None
                and current_open_time != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )
                same_timestamp_amounts = []

            if historical_amounts:
                historical_median = np.median(
                    historical_amounts
                )
            else:
                historical_median = np.nan

            historical_medians.append(
                historical_median
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "historical_median_amount_before_trade"
        ] = historical_medians

        result_frames.append(current)

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "current_to_historical_median_amount_ratio"
    ] = (
        result["amount"]
        / result[
            "historical_median_amount_before_trade"
        ]
    )

    result[
        "historical_size_available"
    ] = (
        result[
            "historical_median_amount_before_trade"
        ].notna()
        & (
            result[
                "historical_median_amount_before_trade"
            ]
            > 0
        )
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(drop=True)
    )

In [148]:
trade_features_table = (
    attach_historical_position_size_features(
        trade_features_table
    )
)

In [149]:
print(
    "Historical size available:",
    trade_features_table[
        "historical_size_available"
    ].sum(),
)

print(
    "Invalid historical medians:",
    (
        trade_features_table[
            "historical_median_amount_before_trade"
        ]
        <= 0
    ).sum(),
)

print(
    "Invalid size ratios:",
    (
        trade_features_table[
            "current_to_historical_median_amount_ratio"
        ]
        <= 0
    ).sum(),
)

Historical size available: 38355
Invalid historical medians: 0
Invalid size ratios: 0


In [150]:
display(
    trade_features_table[
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "open_date_time",
            "amount",
            "historical_median_amount_before_trade",
            "current_to_historical_median_amount_ratio",
            "realized_pnl_before_trade",
            "realized_distance_to_loss_limit",
            "reverse_profit_per_lot",
        ]
    ].head(30)
)

,trade_row_id,account_id,campaign_id,open_date_time,amount,historical_median_amount_before_trade,current_to_historical_median_amount_ratio,realized_pnl_before_trade,realized_distance_to_loss_limit,reverse_profit_per_lot
0,1,D#1589378,33,2026-02-24 12:01:08+00:00,0.01,NaN,NaN,0.00,200.00,-44.0
1,2,D#1589408,33,2026-02-24 02:46:35+00:00,0.43,NaN,NaN,0.00,200.00,395.0
2,3,D#1589408,33,2026-02-24 02:53:40+00:00,0.10,0.430,0.232558,-187.92,12.08,-95.0
3,4,D#1589408,33,2026-02-24 02:57:37+00:00,0.30,0.265,1.132075,-182.62,17.38,198.0
4,5,D#1589410,33,2026-02-25 00:13:04+00:00,0.10,NaN,NaN,0.00,200.00,-496.0
5,6,D#1589410,33,2026-02-25 00:23:39+00:00,0.10,0.100,1.000000,45.40,245.40,-464.0
6,7,D#1589419,33,2026-02-24 05:32:22+00:00,0.01,NaN,NaN,0.00,200.00,1968.0
7,8,D#1589428,33,2026-02-24 02:38:47+00:00,0.10,NaN,NaN,0.00,200.00,-172.0
8,9,D#1589428,33,2026-02-24 07:24:19+00:00,0.15,0.100,1.500000,13.00,213.00,-764.0
9,10,D#1589428,33,2026-02-24 08:38:11+00:00,0.15,0.125,1.200000,121.29,321.29,2187.0


In [151]:
historical_size_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "historical_size_available"
        ]
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "amount",
            "historical_median_amount_before_trade",
            "current_to_historical_median_amount_ratio",
            "realized_pnl_before_trade",
            "realized_distance_to_loss_limit",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [152]:
def learn_large_size_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually large position size.

    Args:
        train_data: Historical training observations.
        candidate_quantiles: Candidate upper-tail size thresholds.
        min_signal_trades: Minimum signal observations.
        min_control_trades: Minimum control observations.

    Returns:
        Best threshold information, or None.
    """
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "current_to_historical_median_amount_ratio"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "current_to_historical_median_amount_ratio"
            ]
            >= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [153]:
def walk_forward_historical_size(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually large current position size.

    Args:
        data: Eligible trade observations.
        minimum_training_campaigns: Historical campaigns required.

    Returns:
        One row per OOS campaign.
    """
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_large_size_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "current_to_historical_median_amount_ratio"
            ]
            >= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [154]:
historical_size_walk_forward = (
    walk_forward_historical_size(
        historical_size_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    historical_size_walk_forward
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,test_uplift_vs_control
0,43,0.7,1.166667,363,812,-29.129132,-66.974302,37.845170
1,44,0.7,1.166667,396,919,12.738973,-18.790207,31.529180
2,45,0.7,1.166667,310,709,19.662315,12.271441,7.390874
3,46,0.7,1.166667,418,695,-8.505622,-22.953957,14.448335
4,47,0.7,1.200000,230,551,-10.144888,0.835027,-10.979915
5,48,0.7,1.200000,367,838,-65.400545,1.930430,-67.330975
6,49,0.7,1.200000,363,732,-82.529773,-15.575273,-66.954500
7,50,0.7,1.200000,291,626,-49.041409,80.272897,-129.314306
8,51,0.7,1.200000,253,696,-50.275889,-8.652360,-41.623529
9,52,0.7,1.200000,267,523,78.469716,-9.923136,88.392852


In [155]:
drawdown_size_test_data = (
    historical_size_test_data.loc[
        historical_size_test_data[
            "realized_pnl_before_trade"
        ] < 0
    ]
    .copy()
)

In [156]:
drawdown_size_walk_forward = (
    walk_forward_historical_size(
        drawdown_size_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    drawdown_size_walk_forward
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,test_uplift_vs_control
0,43,0.7,1.333333,118,245,19.050847,25.982857,-6.932010
1,44,0.7,1.333333,142,337,98.440141,31.029674,67.410467
2,45,0.7,1.333333,88,222,-112.905114,1.869369,-114.774483
3,46,0.7,1.333333,149,272,48.984228,-12.757353,61.741581
4,47,0.7,1.333333,94,206,-48.514088,12.841262,-61.355350
5,48,0.9,2.500000,49,368,-115.663265,20.209783,-135.873048
6,49,0.7,1.333333,118,285,-189.389831,7.477193,-196.867023
7,50,0.9,2.500000,41,338,-181.004878,93.428797,-274.433675
8,51,0.9,2.500000,49,347,-81.253061,23.824084,-105.077145
9,52,0.9,2.500000,48,232,37.500000,69.302586,-31.802586


In [157]:
def summarize_walk_forward(
    results: pd.DataFrame,
    label: str,
) -> None:
    """Prints key OOS walk-forward statistics."""
    valid_uplifts = (
        results[
            "test_uplift_vs_control"
        ]
        .dropna()
    )

    ci_lower, ci_upper = (
        bootstrap_campaign_uplift_ci(
            results
        )
    )

    print(label)
    print(
        "Mean OOS uplift:",
        valid_uplifts.mean(),
    )
    print(
        "Median OOS uplift:",
        valid_uplifts.median(),
    )
    print(
        "Positive campaign share:",
        (valid_uplifts > 0).mean(),
    )
    print(
        "95% bootstrap CI:",
        (
            ci_lower,
            ci_upper,
        ),
    )

In [158]:
summarize_walk_forward(
    historical_size_walk_forward,
    "Large size relative to history",
)

print()

summarize_walk_forward(
    drawdown_size_walk_forward,
    "Large size while in drawdown",
)

Large size relative to history
Mean OOS uplift: -1.9305708346247101
Median OOS uplift: 2.9948830919962326
Positive campaign share: 0.5
95% bootstrap CI: (np.float64(-24.078806905321755), np.float64(18.878528858376264))

Large size while in drawdown
Mean OOS uplift: -25.100298440957307
Median OOS uplift: -11.344382197369786
Positive campaign share: 0.375
95% bootstrap CI: (np.float64(-63.43447707648287), np.float64(8.88296656634157))


# Stop-loss / take-profit behavior

## Create SL/TP features

In [159]:
trade_features_table[
    "has_stop_loss"
] = (
    trade_features_table[
        "sl_price"
    ].notna()
)

trade_features_table[
    "has_take_profit"
] = (
    trade_features_table[
        "tp_price"
    ].notna()
)

trade_features_table[
    "has_both_sl_tp"
] = (
    trade_features_table[
        "has_stop_loss"
    ]
    & trade_features_table[
        "has_take_profit"
    ]
)

trade_features_table[
    "has_neither_sl_tp"
] = (
    ~trade_features_table[
        "has_stop_loss"
    ]
    & ~trade_features_table[
        "has_take_profit"
    ]
)

## Calculate directional SL and TP distances

In [160]:
buy_mask = (
    trade_features_table[
        "side"
    ].str.lower().eq("buy")
)

sell_mask = (
    trade_features_table[
        "side"
    ].str.lower().eq("sell")
)


trade_features_table[
    "stop_loss_distance"
] = np.nan

trade_features_table[
    "take_profit_distance"
] = np.nan


trade_features_table.loc[
    buy_mask
    & trade_features_table[
        "has_stop_loss"
    ],
    "stop_loss_distance",
] = (
    trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "open_price",
    ]
    - trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "sl_price",
    ]
)


trade_features_table.loc[
    sell_mask
    & trade_features_table[
        "has_stop_loss"
    ],
    "stop_loss_distance",
] = (
    trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "sl_price",
    ]
    - trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "open_price",
    ]
)


trade_features_table.loc[
    buy_mask
    & trade_features_table[
        "has_take_profit"
    ],
    "take_profit_distance",
] = (
    trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "tp_price",
    ]
    - trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "open_price",
    ]
)


trade_features_table.loc[
    sell_mask
    & trade_features_table[
        "has_take_profit"
    ],
    "take_profit_distance",
] = (
    trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "open_price",
    ]
    - trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "tp_price",
    ]
)

## Validate the direction

In [161]:
print(
    "Trades with SL:",
    trade_features_table[
        "has_stop_loss"
    ].sum(),
)

print(
    "Trades with TP:",
    trade_features_table[
        "has_take_profit"
    ].sum(),
)

print(
    "Non-positive SL distances:",
    (
        trade_features_table[
            "stop_loss_distance"
        ].notna()
        & (
            trade_features_table[
                "stop_loss_distance"
            ]
            <= 0
        )
    ).sum(),
)

print(
    "Non-positive TP distances:",
    (
        trade_features_table[
            "take_profit_distance"
        ].notna()
        & (
            trade_features_table[
                "take_profit_distance"
            ]
            <= 0
        )
    ).sum(),
)

Trades with SL: 21919
Trades with TP: 25590
Non-positive SL distances: 3207
Non-positive TP distances: 97


## Mark valid distances

In [162]:
trade_features_table[
    "stop_loss_distance_valid"
] = (
    trade_features_table[
        "stop_loss_distance"
    ].gt(0)
)

trade_features_table[
    "take_profit_distance_valid"
] = (
    trade_features_table[
        "take_profit_distance"
    ].gt(0)
)

In [163]:
trade_features_table[
    "valid_stop_loss_distance"
] = (
    trade_features_table[
        "stop_loss_distance"
    ].where(
        trade_features_table[
            "stop_loss_distance_valid"
        ]
    )
)

trade_features_table[
    "valid_take_profit_distance"
] = (
    trade_features_table[
        "take_profit_distance"
    ].where(
        trade_features_table[
            "take_profit_distance_valid"
        ]
    )
)

## Normalize by entry price

In [164]:
trade_features_table[
    "stop_loss_distance_pct"
] = (
    trade_features_table[
        "valid_stop_loss_distance"
    ]
    / trade_features_table[
        "open_price"
    ]
)

trade_features_table[
    "take_profit_distance_pct"
] = (
    trade_features_table[
        "valid_take_profit_distance"
    ]
    / trade_features_table[
        "open_price"
    ]
)

## PLanned reward-to-risk ratio

In [165]:
trade_features_table[
    "planned_reward_to_risk_ratio"
] = (
    trade_features_table[
        "valid_take_profit_distance"
    ]
    / trade_features_table[
        "valid_stop_loss_distance"
    ]
)

trade_features_table[
    "planned_reward_to_risk_ratio"
] = (
    trade_features_table[
        "planned_reward_to_risk_ratio"
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

## Inspect the feature family

In [166]:
sl_tp_feature_columns = [
    "trade_row_id",
    "account_id",
    "campaign_id",
    "side",
    "open_price",
    "sl_price",
    "tp_price",
    "has_stop_loss",
    "has_take_profit",
    "has_both_sl_tp",
    "has_neither_sl_tp",
    "stop_loss_distance",
    "take_profit_distance",
    "stop_loss_distance_pct",
    "take_profit_distance_pct",
    "planned_reward_to_risk_ratio",
    "reverse_profit_per_lot",
]

display(
    trade_features_table[
        sl_tp_feature_columns
    ].head(30)
)

,trade_row_id,account_id,campaign_id,side,open_price,sl_price,tp_price,has_stop_loss,has_take_profit,has_both_sl_tp,has_neither_sl_tp,stop_loss_distance,take_profit_distance,stop_loss_distance_pct,take_profit_distance_pct,planned_reward_to_risk_ratio,reverse_profit_per_lot
0,1,D#1589378,33,sell,5170.87,NaN,NaN,False,False,False,True,NaN,NaN,NaN,NaN,NaN,-44.0
1,2,D#1589408,33,sell,5185.05,5188.98,5183.56,True,True,True,False,3.93,1.49,0.000758,0.000287,0.379135,395.0
2,3,D#1589408,33,buy,5185.71,NaN,NaN,False,False,False,True,NaN,NaN,NaN,NaN,NaN,-95.0
3,4,D#1589408,33,sell,5186.38,NaN,5183.80,False,True,False,False,NaN,2.58,NaN,0.000497,NaN,198.0
4,5,D#1589410,33,buy,5139.73,NaN,NaN,False,False,False,True,NaN,NaN,NaN,NaN,NaN,-496.0
5,6,D#1589410,33,buy,5146.41,NaN,5200.00,False,True,False,False,NaN,53.59,NaN,0.010413,NaN,-464.0
6,7,D#1589419,33,buy,5170.72,NaN,NaN,False,False,False,True,NaN,NaN,NaN,NaN,NaN,1968.0
7,8,D#1589428,33,buy,5185.88,NaN,NaN,False,False,False,True,NaN,NaN,NaN,NaN,NaN,-172.0
8,9,D#1589428,33,buy,5175.73,5176.34,5183.05,True,True,True,False,-0.61,7.32,NaN,0.001414,NaN,-764.0
9,10,D#1589428,33,buy,5181.41,5139.10,5188.66,True,True,True,False,42.31,7.25,0.008166,0.001399,0.171354,2187.0


In [167]:
display(
    trade_features_table[
        [
            "stop_loss_distance_pct",
            "take_profit_distance_pct",
            "planned_reward_to_risk_ratio",
        ]
    ].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

,stop_loss_distance_pct,take_profit_distance_pct,planned_reward_to_risk_ratio
count,18712.000000,25493.000000,16080.000000
mean,0.002779,0.005266,8.346183
std,0.093644,0.138207,109.965206
min,0.000002,0.000004,0.000051
1%,0.000015,0.000195,0.150612
5%,0.000176,0.000424,0.505931
10%,0.000296,0.000567,0.740371
25%,0.000591,0.001023,1.159150
50%,0.001099,0.001936,1.948718
75%,0.001980,0.003332,3.115194


In [168]:
def walk_forward_binary_feature(
    data: pd.DataFrame,
    feature_column: str,
    signal_value: bool,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Evaluates a binary feature walk-forward by campaign.

    Args:
        data: Trade-level feature data.
        feature_column: Binary feature to evaluate.
        signal_value: Value defining the signal group.
        minimum_training_campaigns: Number of initial campaigns
            reserved before OOS evaluation begins.

    Returns:
        One row per OOS campaign.
    """
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        test_campaign = campaign_order[
            test_index
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        signal_mask = (
            test_data[
                feature_column
            ].eq(signal_value)
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "feature": feature_column,
                "signal_value": signal_value,
                "test_signal_trades": len(
                    signal_data
                ),
                "test_control_trades": len(
                    control_data
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [169]:
sl_tp_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "has_stop_loss",
            "has_take_profit",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [170]:
no_stop_loss_walk_forward = (
    walk_forward_binary_feature(
        data=sl_tp_test_data,
        feature_column="has_stop_loss",
        signal_value=False,
        minimum_training_campaigns=10,
    )
)

display(
    no_stop_loss_walk_forward
)

summarize_walk_forward(
    no_stop_loss_walk_forward,
    "No stop-loss",
)

,test_campaign,feature,signal_value,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,test_uplift_vs_control
0,43,has_stop_loss,False,764,640,-61.825098,-82.353203,20.528105
1,44,has_stop_loss,False,864,713,-11.870910,19.555259,-31.426170
2,45,has_stop_loss,False,622,633,64.239258,-21.474969,85.714226
3,46,has_stop_loss,False,787,603,-4.004045,-93.357380,89.353335
4,47,has_stop_loss,False,505,481,-6.103960,-10.677597,4.573637
5,48,has_stop_loss,False,853,616,-0.903869,-13.943669,13.039800
6,49,has_stop_loss,False,631,736,28.846804,-133.984872,162.831676
7,50,has_stop_loss,False,566,605,78.266078,13.043609,65.222469
8,51,has_stop_loss,False,619,546,-34.345557,26.743694,-61.089252
9,52,has_stop_loss,False,509,478,27.874263,-48.456037,76.330300


No stop-loss
Mean OOS uplift: 18.2109456193689
Median OOS uplift: 15.421613066831863
Positive campaign share: 0.6666666666666666
95% bootstrap CI: (np.float64(-2.721861603949466), np.float64(40.02898492406942))


In [171]:
no_take_profit_walk_forward = (
    walk_forward_binary_feature(
        data=sl_tp_test_data,
        feature_column="has_take_profit",
        signal_value=False,
        minimum_training_campaigns=10,
    )
)

display(
    no_take_profit_walk_forward
)

summarize_walk_forward(
    no_take_profit_walk_forward,
    "No take-profit",
)

,test_campaign,feature,signal_value,test_signal_trades,test_control_trades,signal_mean_reverse_profit_per_lot,control_mean_reverse_profit_per_lot,test_uplift_vs_control
0,43,has_take_profit,False,676,728,17.154031,-153.209547,170.363578
1,44,has_take_profit,False,708,869,2.997928,1.799655,1.198274
2,45,has_take_profit,False,496,759,18.751277,22.480276,-3.728999
3,46,has_take_profit,False,662,728,-28.809945,-55.458104,26.648160
4,47,has_take_profit,False,349,637,-4.985960,-10.170054,5.184094
5,48,has_take_profit,False,652,817,18.915644,-26.552387,45.468031
6,49,has_take_profit,False,582,785,30.315909,-124.910053,155.225962
7,50,has_take_profit,False,490,681,102.109150,3.166667,98.942483
8,51,has_take_profit,False,480,685,-8.478318,-3.778467,-4.699851
9,52,has_take_profit,False,450,537,79.743365,-83.535382,163.278747


No take-profit
Mean OOS uplift: 36.11523305402053
Median OOS uplift: 22.529411062528716
Positive campaign share: 0.7083333333333334
95% bootstrap CI: (np.float64(11.172117652050947), np.float64(63.49814480864117))


In [172]:
trade_features_table[
    "sl_tp_configuration"
] = np.select(
    [
        (
            trade_features_table["has_stop_loss"]
            & trade_features_table["has_take_profit"]
        ),
        (
            trade_features_table["has_stop_loss"]
            & ~trade_features_table["has_take_profit"]
        ),
        (
            ~trade_features_table["has_stop_loss"]
            & trade_features_table["has_take_profit"]
        ),
        (
            ~trade_features_table["has_stop_loss"]
            & ~trade_features_table["has_take_profit"]
        ),
    ],
    [
        "both_sl_tp",
        "sl_only",
        "tp_only",
        "neither_sl_tp",
    ],
    default="unknown",
)

In [173]:
display(
    trade_features_table[
        "sl_tp_configuration"
    ]
    .value_counts()
    .rename("trade_count")
    .reset_index()
)

,sl_tp_configuration,trade_count
0,both_sl_tp,18729
1,neither_sl_tp,17740
2,tp_only,6861
3,sl_only,3190


In [174]:
def walk_forward_sl_tp_configuration(
    data: pd.DataFrame,
    signal_group: str,
    control_group: str = "both_sl_tp",
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Evaluates an SL/TP configuration against a reference group.

    Args:
        data: Trade-level feature data.
        signal_group: SL/TP configuration being tested.
        control_group: Reference SL/TP configuration.
        minimum_training_campaigns: Initial campaigns excluded
            from OOS evaluation.

    Returns:
        One row per OOS campaign.
    """
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        test_campaign = campaign_order[
            test_index
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ]

        signal_data = test_data.loc[
            test_data[
                "sl_tp_configuration"
            ].eq(signal_group)
        ]

        control_data = test_data.loc[
            test_data[
                "sl_tp_configuration"
            ].eq(control_group)
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "signal_group": signal_group,
                "control_group": control_group,
                "test_signal_trades": len(
                    signal_data
                ),
                "test_control_trades": len(
                    control_data
                ),
                "signal_mean_reverse_profit_per_lot": (
                    signal_mean
                ),
                "control_mean_reverse_profit_per_lot": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [175]:
sl_tp_configuration_data = (
    trade_features_table.loc[
        trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "sl_tp_configuration",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [176]:
sl_only_walk_forward = (
    walk_forward_sl_tp_configuration(
        data=sl_tp_configuration_data,
        signal_group="sl_only",
    )
)

summarize_walk_forward(
    sl_only_walk_forward,
    "SL only vs both SL + TP",
)

SL only vs both SL + TP
Mean OOS uplift: 83.30350665409912
Median OOS uplift: 66.17572449021536
Positive campaign share: 0.875
95% bootstrap CI: (np.float64(40.68816723908543), np.float64(132.84761067791007))


In [177]:
tp_only_walk_forward = (
    walk_forward_sl_tp_configuration(
        data=sl_tp_configuration_data,
        signal_group="tp_only",
    )
)

summarize_walk_forward(
    tp_only_walk_forward,
    "TP only vs both SL + TP",
)

TP only vs both SL + TP
Mean OOS uplift: 21.726522889703286
Median OOS uplift: 7.502833992994699
Positive campaign share: 0.5416666666666666
95% bootstrap CI: (np.float64(-14.5255506953401), np.float64(57.86157706920439))


In [178]:
neither_sl_tp_walk_forward = (
    walk_forward_sl_tp_configuration(
        data=sl_tp_configuration_data,
        signal_group="neither_sl_tp",
    )
)

summarize_walk_forward(
    neither_sl_tp_walk_forward,
    "Neither SL nor TP vs both SL + TP",
)

Neither SL nor TP vs both SL + TP
Mean OOS uplift: 35.38821426002471
Median OOS uplift: 29.01710037588461
Positive campaign share: 0.6666666666666666
95% bootstrap CI: (np.float64(9.356110092616214), np.float64(63.996318972863804))


In [179]:
take_profit_distance_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "valid_take_profit_distance"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "take_profit_distance_pct",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

print(
    "Eligible TP-distance trades:",
    len(take_profit_distance_test_data),
)

display(
    take_profit_distance_test_data[
        "take_profit_distance_pct"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)

Eligible TP-distance trades: 25493


count    25493.000000
mean         0.005266
std          0.138207
min          0.000004
10%          0.000567
25%          0.001023
50%          0.001936
75%          0.003332
90%          0.006510
max          9.110184
Name: take_profit_distance_pct, dtype: float64

In [180]:
def learn_wide_take_profit_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually wide take-profit placement."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "take_profit_distance_pct"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "take_profit_distance_pct"
            ]
            >= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [181]:
def walk_forward_take_profit_distance(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually wide take-profit placement."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_wide_take_profit_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "take_profit_distance_pct"
            ]
            >= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [182]:
take_profit_distance_walk_forward = (
    walk_forward_take_profit_distance(
        take_profit_distance_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    take_profit_distance_walk_forward
)

summarize_walk_forward(
    take_profit_distance_walk_forward,
    "Wide take-profit distance",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.9,0.008581,66,660,-285.677727
1,44,0.9,0.008547,40,824,-82.298908
2,45,0.9,0.008201,48,707,53.239468
3,46,0.9,0.008038,37,689,-1169.105813
4,47,0.5,0.002068,287,346,52.570510
5,48,0.5,0.002060,324,492,-9.803290
6,49,0.5,0.002030,331,454,-259.320300
7,50,0.6,0.002420,296,381,50.151521
8,51,0.6,0.002428,194,487,39.911855
9,52,0.6,0.002400,247,289,-59.846444


Wide take-profit distance
Mean OOS uplift: -68.07538416713201
Median OOS uplift: -1.5097173423055632
Positive campaign share: 0.5
95% bootstrap CI: (np.float64(-184.76944361103884), np.float64(11.831496495755383))


In [183]:
stop_loss_distance_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "valid_stop_loss_distance"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "stop_loss_distance_pct",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

print(
    "Eligible SL-distance trades:",
    len(stop_loss_distance_test_data),
)

display(
    stop_loss_distance_test_data[
        "stop_loss_distance_pct"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)

Eligible SL-distance trades: 18712


count    18712.000000
mean         0.002779
std          0.093644
min          0.000002
10%          0.000296
25%          0.000591
50%          0.001099
75%          0.001980
90%          0.003307
max          9.017238
Name: stop_loss_distance_pct, dtype: float64

In [184]:
def learn_wide_stop_loss_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually wide stop-loss placement."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "stop_loss_distance_pct"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "stop_loss_distance_pct"
            ]
            >= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [185]:
def walk_forward_stop_loss_distance(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually wide stop-loss placement."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_wide_stop_loss_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "stop_loss_distance_pct"
            ]
            >= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [186]:
stop_loss_distance_walk_forward = (
    walk_forward_stop_loss_distance(
        stop_loss_distance_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    stop_loss_distance_walk_forward
)

summarize_walk_forward(
    stop_loss_distance_walk_forward,
    "Wide stop-loss distance",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.9,0.003761,49,481,230.135661
1,44,0.9,0.003752,41,575,150.960933
2,45,0.9,0.003684,36,488,-48.622719
3,46,0.9,0.003637,38,484,-308.417464
4,47,0.9,0.003595,51,394,-156.954416
5,48,0.9,0.003616,54,480,465.327130
6,49,0.9,0.003616,42,584,166.552027
7,50,0.9,0.003576,32,472,282.881285
8,51,0.9,0.003536,32,467,-7.787406
9,52,0.9,0.003505,39,359,287.362903


Wide stop-loss distance
Mean OOS uplift: 173.64257294646157
Median OOS uplift: 141.2340062502662
Positive campaign share: 0.75
95% bootstrap CI: (np.float64(77.28368666217071), np.float64(270.85017845038647))


In [187]:
display(
    stop_loss_distance_walk_forward[
        [
            "test_campaign",
            "learned_quantile",
            "learned_threshold",
            "test_signal_trades",
            "test_control_trades",
            "test_uplift_vs_control",
        ]
    ]
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.9,0.003761,49,481,230.135661
1,44,0.9,0.003752,41,575,150.960933
2,45,0.9,0.003684,36,488,-48.622719
3,46,0.9,0.003637,38,484,-308.417464
4,47,0.9,0.003595,51,394,-156.954416
5,48,0.9,0.003616,54,480,465.327130
6,49,0.9,0.003616,42,584,166.552027
7,50,0.9,0.003576,32,472,282.881285
8,51,0.9,0.003536,32,467,-7.787406
9,52,0.9,0.003505,39,359,287.362903


In [188]:
display(
    stop_loss_distance_walk_forward[
        [
            "learned_quantile",
            "learned_threshold",
        ]
    ].describe()
)

,learned_quantile,learned_threshold
count,24.0,24.000000
mean,0.9,0.003483
std,0.0,0.000145
min,0.9,0.003316
25%,0.9,0.003347
50%,0.9,0.003479
75%,0.9,0.003600
max,0.9,0.003761


In [189]:
display(
    stop_loss_distance_walk_forward[
        "learned_quantile"
    ]
    .value_counts()
    .sort_index()
    .rename("campaign_count")
    .reset_index()
)

,learned_quantile,campaign_count
0,0.9,24


In [190]:
median_entry_price = (
    trade_features_table.loc[
        trade_features_table[
            "campaign_id"
        ].isin(
            stop_loss_distance_walk_forward[
                "test_campaign"
            ]
        ),
        "open_price",
    ]
    .median()
)

print(
    "Approximate median entry price:",
    median_entry_price,
)

Approximate median entry price: 4518.82


In [191]:
reward_to_risk_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "planned_reward_to_risk_ratio"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "planned_reward_to_risk_ratio",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

print(
    "Eligible reward-to-risk trades:",
    len(reward_to_risk_test_data),
)

display(
    reward_to_risk_test_data[
        "planned_reward_to_risk_ratio"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
)

Eligible reward-to-risk trades: 16080


count    16080.000000
mean         8.346183
std        109.965206
min          0.000051
10%          0.740371
25%          1.159150
50%          1.948718
75%          3.115194
90%          5.841457
max       9997.181818
Name: planned_reward_to_risk_ratio, dtype: float64

In [192]:
def learn_high_reward_to_risk_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually high reward-to-risk."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "planned_reward_to_risk_ratio"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "planned_reward_to_risk_ratio"
            ]
            >= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [193]:
def walk_forward_reward_to_risk(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually high planned reward-to-risk."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_high_reward_to_risk_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "planned_reward_to_risk_ratio"
            ]
            >= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [194]:
reward_to_risk_walk_forward = (
    walk_forward_reward_to_risk(
        reward_to_risk_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    reward_to_risk_walk_forward
)

summarize_walk_forward(
    reward_to_risk_walk_forward,
    "High planned reward-to-risk",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.5,2.055046,209,207,-224.134028
1,44,0.5,2.057292,260,285,16.946343
2,45,0.5,2.049014,230,238,50.319015
3,46,0.5,2.047724,211,245,-71.188742
4,47,0.5,2.038840,184,214,-45.358379
5,48,0.5,2.027088,207,249,97.521890
6,49,0.5,2.018233,250,247,-91.226842
7,50,0.5,2.018909,208,194,2.615719
8,51,0.5,2.021757,206,229,-76.647836
9,52,0.5,2.017489,173,162,-59.714158


High planned reward-to-risk
Mean OOS uplift: -38.101574375440606
Median OOS uplift: -40.66631213492626
Positive campaign share: 0.375
95% bootstrap CI: (np.float64(-69.54419025293586), np.float64(-7.021722114009514))


# Historical performance

In [195]:
idea_performance = (
    idea_features[
        [
            "idea_id",
            "account_id",
            "campaign_id",
            "idea_start_time",
            "idea_end_time",
            "total_amount",
            "total_net_profit",
            "is_profitable_idea",
            "is_losing_idea",
            "is_breakeven_idea",
        ]
    ]
    .copy()
)

idea_performance[
    "idea_profit_per_lot"
] = (
    idea_performance["total_net_profit"]
    / idea_performance["total_amount"]
)

In [196]:
idea_performance[
    "historical_win_count_component"
] = (
    idea_performance[
        "is_profitable_idea"
    ].astype(int)
)

idea_performance[
    "historical_loss_count_component"
] = (
    idea_performance[
        "is_losing_idea"
    ].astype(int)
)

idea_performance[
    "historical_idea_count_component"
] = 1

idea_performance[
    "historical_win_profit_component"
] = (
    idea_performance[
        "idea_profit_per_lot"
    ].where(
        idea_performance[
            "is_profitable_idea"
        ],
        0.0,
    )
)

idea_performance[
    "historical_loss_abs_profit_component"
] = (
    idea_performance[
        "idea_profit_per_lot"
    ]
    .abs()
    .where(
        idea_performance[
            "is_losing_idea"
        ],
        0.0,
    )
)

In [197]:
historical_performance_events = (
    idea_performance
    .groupby(
        [
            "account_id",
            "idea_end_time",
        ],
        as_index=False,
    )
    .agg(
        completed_idea_count=(
            "historical_idea_count_component",
            "sum",
        ),
        completed_win_count=(
            "historical_win_count_component",
            "sum",
        ),
        completed_loss_count=(
            "historical_loss_count_component",
            "sum",
        ),
        completed_win_profit_sum=(
            "historical_win_profit_component",
            "sum",
        ),
        completed_loss_abs_profit_sum=(
            "historical_loss_abs_profit_component",
            "sum",
        ),
    )
)

In [198]:
historical_performance_events = (
    historical_performance_events
    .sort_values(
        [
            "account_id",
            "idea_end_time",
        ]
    )
    .reset_index(drop=True)
)

In [199]:
historical_performance_events[
    "past_completed_idea_count"
] = (
    historical_performance_events
    .groupby("account_id")[
        "completed_idea_count"
    ]
    .cumsum()
)

historical_performance_events[
    "past_winning_idea_count"
] = (
    historical_performance_events
    .groupby("account_id")[
        "completed_win_count"
    ]
    .cumsum()
)

historical_performance_events[
    "past_losing_idea_count"
] = (
    historical_performance_events
    .groupby("account_id")[
        "completed_loss_count"
    ]
    .cumsum()
)

historical_performance_events[
    "past_win_profit_sum"
] = (
    historical_performance_events
    .groupby("account_id")[
        "completed_win_profit_sum"
    ]
    .cumsum()
)

historical_performance_events[
    "past_loss_abs_profit_sum"
] = (
    historical_performance_events
    .groupby("account_id")[
        "completed_loss_abs_profit_sum"
    ]
    .cumsum()
)

In [200]:
current_ideas = (
    idea_performance[
        [
            "idea_id",
            "account_id",
            "campaign_id",
            "idea_start_time",
        ]
    ]
    .copy()
)

In [201]:
current_ideas = (
    current_ideas
    .sort_values(
        [
            "idea_start_time",
            "account_id",
        ]
    )
    .reset_index(drop=True)
)

historical_performance_events = (
    historical_performance_events
    .sort_values(
        [
            "idea_end_time",
            "account_id",
        ]
    )
    .reset_index(drop=True)
)

In [202]:
historical_idea_features = (
    pd.merge_asof(
        current_ideas,
        historical_performance_events[
            [
                "account_id",
                "idea_end_time",
                "past_completed_idea_count",
                "past_winning_idea_count",
                "past_losing_idea_count",
                "past_win_profit_sum",
                "past_loss_abs_profit_sum",
            ]
        ],
        left_on="idea_start_time",
        right_on="idea_end_time",
        by="account_id",
        direction="backward",
        allow_exact_matches=False,
    )
)

In [203]:
historical_idea_features[
    "past_idea_win_rate"
] = (
    historical_idea_features[
        "past_winning_idea_count"
    ]
    / historical_idea_features[
        "past_completed_idea_count"
    ]
)

In [204]:
historical_idea_features[
    "past_mean_win_profit_per_lot"
] = (
    historical_idea_features[
        "past_win_profit_sum"
    ]
    / historical_idea_features[
        "past_winning_idea_count"
    ]
)

In [205]:
historical_idea_features[
    "past_mean_loss_abs_profit_per_lot"
] = (
    historical_idea_features[
        "past_loss_abs_profit_sum"
    ]
    / historical_idea_features[
        "past_losing_idea_count"
    ]
)

In [206]:
historical_idea_features[
    "past_payoff_ratio"
] = (
    historical_idea_features[
        "past_mean_win_profit_per_lot"
    ]
    / historical_idea_features[
        "past_mean_loss_abs_profit_per_lot"
    ]
)

In [207]:
historical_idea_features[
    "past_win_rate_available"
] = (
    historical_idea_features[
        "past_completed_idea_count"
    ] >= 1
)

historical_idea_features[
    "past_payoff_ratio_available"
] = (
    (
        historical_idea_features[
            "past_winning_idea_count"
        ] >= 1
    )
    & (
        historical_idea_features[
            "past_losing_idea_count"
        ] >= 1
    )
)

In [208]:
historical_idea_features[
    "past_performance_robust_history"
] = (
    (
        historical_idea_features[
            "past_winning_idea_count"
        ] >= 2
    )
    & (
        historical_idea_features[
            "past_losing_idea_count"
        ] >= 2
    )
)

In [209]:
display(
    historical_idea_features[
        [
            "idea_id",
            "account_id",
            "campaign_id",
            "idea_start_time",
            "idea_end_time",
            "past_completed_idea_count",
            "past_winning_idea_count",
            "past_losing_idea_count",
            "past_idea_win_rate",
            "past_mean_win_profit_per_lot",
            "past_mean_loss_abs_profit_per_lot",
            "past_payoff_ratio",
        ]
    ].head(20)
)

,idea_id,account_id,campaign_id,idea_start_time,idea_end_time,past_completed_idea_count,past_winning_idea_count,past_losing_idea_count,past_idea_win_rate,past_mean_win_profit_per_lot,past_mean_loss_abs_profit_per_lot,past_payoff_ratio
0,D#1645625_33_sell_1,D#1645625,33,2026-02-24 00:49:38+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D#1645651_33_sell_1,D#1645651,33,2026-02-24 01:02:15+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,D#1702379_33_buy_1,D#1702379,33,2026-02-24 01:02:19+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,D#1702617_33_buy_1,D#1702617,33,2026-02-24 01:06:23+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,D#1670922_33_sell_1,D#1670922,33,2026-02-24 01:07:03+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,D#1670952_33_sell_1,D#1670952,33,2026-02-24 01:09:51+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,D#1702376_33_buy_1,D#1702376,33,2026-02-24 01:10:51+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,D#1702513_33_sell_1,D#1702513,33,2026-02-24 01:12:05+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,D#1670923_33_buy_1,D#1670923,33,2026-02-24 01:13:46+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,D#1670979_33_sell_1,D#1670979,33,2026-02-24 01:14:40+00:00,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [210]:
print(
    "Ideas with win-rate history:",
    historical_idea_features[
        "past_win_rate_available"
    ].sum(),
)

print(
    "Ideas with payoff-ratio history:",
    historical_idea_features[
        "past_payoff_ratio_available"
    ].sum(),
)

print(
    "Ideas with robust payoff history:",
    historical_idea_features[
        "past_performance_robust_history"
    ].sum(),
)

Ideas with win-rate history: 32865
Ideas with payoff-ratio history: 31614
Ideas with robust payoff history: 30255


In [211]:
invalid_history_matches = (
    historical_idea_features.loc[
        historical_idea_features["idea_end_time"].notna()
        & (
            historical_idea_features["idea_end_time"]
            >= historical_idea_features["idea_start_time"]
        )
    ]
)

print(
    "Invalid historical matches:",
    len(invalid_history_matches),
)

Invalid historical matches: 0


In [212]:
display(
    historical_idea_features.loc[
        historical_idea_features[
            "past_completed_idea_count"
        ].notna(),
        [
            "idea_id",
            "account_id",
            "campaign_id",
            "idea_start_time",
            "idea_end_time",
            "past_completed_idea_count",
            "past_winning_idea_count",
            "past_losing_idea_count",
            "past_idea_win_rate",
            "past_mean_win_profit_per_lot",
            "past_mean_loss_abs_profit_per_lot",
            "past_payoff_ratio",
        ],
    ]
    .sample(
        n=min(
            10,
            historical_idea_features[
                "past_completed_idea_count"
            ].notna().sum(),
        ),
        random_state=42,
    )
)

,idea_id,account_id,campaign_id,idea_start_time,idea_end_time,past_completed_idea_count,past_winning_idea_count,past_losing_idea_count,past_idea_win_rate,past_mean_win_profit_per_lot,past_mean_loss_abs_profit_per_lot,past_payoff_ratio
14138,D#1702462_48_sell_2,D#1702462,48,2026-04-28 09:58:30+00:00,2026-04-28 09:58:06+00:00,29.0,12.0,17.0,0.413793,1394.641667,751.372020,1.856127
4190,D#1702453_38_sell_2,D#1702453,38,2026-03-17 05:43:47+00:00,2026-03-17 05:30:38+00:00,8.0,7.0,1.0,0.875000,746.293862,10.020408,74.477392
32234,D#1702391_66_sell_2,D#1702391,66,2026-07-03 03:30:04+00:00,2026-07-03 02:57:18+00:00,57.0,27.0,30.0,0.473684,647.188796,843.094961,0.767635
6389,D#1702709_40_buy_1,D#1702709,40,2026-03-24 13:00:36+00:00,2026-03-13 18:23:21+00:00,13.0,1.0,12.0,0.076923,1367.800000,323.225000,4.231727
20457,D#1702448_55_buy_1,D#1702448,55,2026-05-26 09:47:53+00:00,2026-05-26 09:47:06+00:00,27.0,8.0,19.0,0.296296,540.661111,836.093222,0.646652
18381,D#1702678_53_buy_2,D#1702678,53,2026-05-19 09:21:15+00:00,2026-05-19 09:16:11+00:00,41.0,23.0,18.0,0.560976,576.614422,627.928671,0.918280
21277,D#1671080_56_buy_2,D#1671080,56,2026-05-29 06:43:06+00:00,2026-05-29 04:23:34+00:00,69.0,37.0,32.0,0.536232,1013.167503,945.138561,1.071978
16281,D#1702383_50_buy_9,D#1702383,50,2026-05-07 16:31:51+00:00,2026-05-07 15:26:41+00:00,21.0,8.0,13.0,0.380952,548.991929,866.279565,0.633735
5963,D#1702645_40_sell_5,D#1702645,40,2026-03-24 05:19:02+00:00,2026-03-24 05:13:10+00:00,20.0,11.0,9.0,0.550000,447.560101,437.656790,1.022628
31006,D#1702641_65_buy_4,D#1702641,65,2026-06-30 03:03:04+00:00,2026-06-30 02:56:43+00:00,36.0,14.0,22.0,0.388889,532.338347,440.837497,1.207561


In [213]:
historical_performance_features = (
    historical_idea_features[
        [
            "idea_id",
            "past_completed_idea_count",
            "past_winning_idea_count",
            "past_losing_idea_count",
            "past_idea_win_rate",
            "past_mean_win_profit_per_lot",
            "past_mean_loss_abs_profit_per_lot",
            "past_payoff_ratio",
            "past_win_rate_available",
            "past_payoff_ratio_available",
            "past_performance_robust_history",
        ]
    ]
    .copy()
)

In [214]:
trade_features_table = (
    trade_features_table
    .merge(
        historical_performance_features,
        on="idea_id",
        how="left",
        validate="many_to_one",
    )
)

In [215]:
print(
    "Trade feature rows:",
    len(trade_features_table),
)

print(
    "Unique trade rows:",
    trade_features_table[
        "trade_row_id"
    ].nunique(),
)

Trade feature rows: 46520
Unique trade rows: 46520


In [216]:
performance_availability_summary = pd.Series(
    {
        "total_trades": len(
            trade_features_table
        ),
        "win_rate_available": (
            trade_features_table[
                "past_win_rate_available"
            ].fillna(False).sum()
        ),
        "payoff_ratio_available": (
            trade_features_table[
                "past_payoff_ratio_available"
            ].fillna(False).sum()
        ),
        "robust_payoff_history": (
            trade_features_table[
                "past_performance_robust_history"
            ].fillna(False).sum()
        ),
    }
)

display(
    performance_availability_summary
)

total_trades              46520
win_rate_available        45756
payoff_ratio_available    44046
robust_payoff_history     42077
dtype: int64

In [217]:
historical_win_rate_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_idea_win_rate"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_idea_win_rate",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [218]:
def learn_low_win_rate_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually low historical win rate."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "past_idea_win_rate"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "past_idea_win_rate"
            ]
            <= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [219]:
def walk_forward_low_historical_win_rate(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually low historical win rate."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_low_win_rate_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "past_idea_win_rate"
            ]
            <= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [220]:
historical_win_rate_walk_forward = (
    walk_forward_low_historical_win_rate(
        historical_win_rate_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    historical_win_rate_walk_forward
)

summarize_walk_forward(
    historical_win_rate_walk_forward,
    "Low historical win rate",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.1,0.250000,70,1333,78.396204
1,44,0.1,0.250000,62,1511,101.383565
2,45,0.1,0.250000,59,1196,-41.073020
3,46,0.2,0.361111,264,1126,77.531943
4,47,0.2,0.363636,173,813,47.900271
5,48,0.2,0.363636,237,1232,90.152927
6,49,0.1,0.285714,93,1274,-73.203144
7,50,0.3,0.428571,381,790,-66.108076
8,51,0.3,0.428571,328,837,-103.716428
9,52,0.3,0.428571,303,684,-42.599262


Low historical win rate
Mean OOS uplift: -2.4402024951275765
Median OOS uplift: -5.356596058666602
Positive campaign share: 0.4583333333333333
95% bootstrap CI: (np.float64(-25.8544646158942), np.float64(20.541175911339455))


In [221]:
historical_payoff_ratio_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_payoff_ratio"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_payoff_ratio",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [222]:
def learn_low_payoff_ratio_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually low historical payoff ratio."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "past_payoff_ratio"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "past_payoff_ratio"
            ]
            <= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [223]:
def walk_forward_low_historical_payoff_ratio(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually low historical payoff ratio."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_low_payoff_ratio_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "past_payoff_ratio"
            ]
            <= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [224]:
historical_payoff_ratio_walk_forward = (
    walk_forward_low_historical_payoff_ratio(
        historical_payoff_ratio_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    historical_payoff_ratio_walk_forward
)

summarize_walk_forward(
    historical_payoff_ratio_walk_forward,
    "Low historical payoff ratio",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.1,0.257757,76,1300,-11.774127
1,44,0.1,0.265698,113,1442,12.766604
2,45,0.1,0.271907,103,1141,192.089193
3,46,0.1,0.274536,48,1340,-77.542680
4,47,0.1,0.284675,34,950,197.542118
5,48,0.1,0.290502,21,1446,70.312102
6,49,0.1,0.300139,40,1327,-122.254685
7,50,0.1,0.310934,34,1137,-41.963615
8,51,0.1,0.317271,51,1112,-20.488803
9,52,0.1,0.324942,18,969,45.590285


Low historical payoff ratio
Mean OOS uplift: 5.217508300521082
Median OOS uplift: -9.55354406834413
Positive campaign share: 0.4166666666666667
95% bootstrap CI: (np.float64(-36.29965374509583), np.float64(48.85778943609745))


In [225]:
historical_loss_magnitude_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_mean_loss_abs_profit_per_lot"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_mean_loss_abs_profit_per_lot",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [226]:
def learn_high_historical_loss_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually large historical losses."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "past_mean_loss_abs_profit_per_lot"
        ].quantile(quantile)

        signal_mask = (
            train_data[
                "past_mean_loss_abs_profit_per_lot"
            ]
            >= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result["training_uplift"],
    )

In [227]:
def walk_forward_high_historical_loss(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests unusually large historical loss magnitude."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = campaign_order[:test_index]
        test_campaign = campaign_order[test_index]

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_high_historical_loss_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = threshold_result[
            "threshold"
        ]

        signal_mask = (
            test_data[
                "past_mean_loss_abs_profit_per_lot"
            ]
            >= threshold
        )

        signal_data = test_data.loc[
            signal_mask
        ]

        control_data = test_data.loc[
            ~signal_mask
        ]

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result["quantile"]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": len(
                    signal_data
                ),
                "test_control_trades": len(
                    control_data
                ),
                "test_uplift_vs_control": (
                    signal_mean - control_mean
                    if (
                        not np.isnan(signal_mean)
                        and not np.isnan(control_mean)
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [228]:
historical_loss_magnitude_walk_forward = (
    walk_forward_high_historical_loss(
        historical_loss_magnitude_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    historical_loss_magnitude_walk_forward
)

summarize_walk_forward(
    historical_loss_magnitude_walk_forward,
    "High historical loss magnitude",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.5,630.500000,902,496,-17.383432
1,44,0.5,647.607778,837,731,35.907568
2,45,0.5,653.657576,724,530,11.630373
3,46,0.7,922.785714,397,993,-91.986966
4,47,0.6,772.263866,334,652,33.148725
5,48,0.7,914.092000,444,1025,67.494005
6,49,0.6,772.206209,532,835,-2.382100
7,50,0.7,913.648017,313,858,45.091385
8,51,0.6,774.042728,401,764,33.485238
9,52,0.7,904.641071,165,822,118.146079


High historical loss magnitude
Mean OOS uplift: 6.632897903636192
Median OOS uplift: 14.681693400066234
Positive campaign share: 0.5833333333333334
95% bootstrap CI: (np.float64(-15.502521094588571), np.float64(28.491182743890928))


# Position sizing consistency

In [229]:
def attach_historical_sizing_consistency_features(
    target_table: pd.DataFrame,
) -> pd.DataFrame:
    """Adds historical position-sizing consistency features.

    Historical sizing uses only trades opened strictly before the
    current trade within the same account and campaign. Trades opened
    at the same timestamp do not observe one another.

    Args:
        target_table: One-row-per-trade feature table.

    Returns:
        Feature table with historical sizing variability features.
    """
    result_frames = []

    for _, group in target_table.groupby(
        [
            "campaign_id",
            "account_id",
        ],
        sort=False,
        dropna=False,
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        past_counts = []
        past_means = []
        past_stds = []
        past_cvs = []

        for row in current.itertuples():
            current_open_time = row.open_date_time

            if (
                previous_open_time is not None
                and current_open_time != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )
                same_timestamp_amounts = []

            past_count = len(
                historical_amounts
            )

            if past_count >= 1:
                past_mean = np.mean(
                    historical_amounts
                )
            else:
                past_mean = np.nan

            if past_count >= 2:
                past_std = np.std(
                    historical_amounts,
                    ddof=0,
                )
            else:
                past_std = np.nan

            if (
                past_count >= 2
                and past_mean > 0
            ):
                past_cv = (
                    past_std / past_mean
                )
            else:
                past_cv = np.nan

            past_counts.append(
                past_count
            )
            past_means.append(
                past_mean
            )
            past_stds.append(
                past_std
            )
            past_cvs.append(
                past_cv
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "past_amount_count"
        ] = past_counts

        current[
            "past_amount_mean"
        ] = past_means

        current[
            "past_amount_std"
        ] = past_stds

        current[
            "past_amount_cv"
        ] = past_cvs

        current[
            "past_amount_cv_available"
        ] = (
            current[
                "past_amount_count"
            ] >= 2
        )

        result_frames.append(
            current
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(drop=True)
    )

In [230]:
trade_features_table = (
    attach_historical_sizing_consistency_features(
        trade_features_table
    )
)

In [231]:
print(
    "Historical sizing CV available:",
    trade_features_table[
        "past_amount_cv_available"
    ].sum(),
)

print(
    "Invalid CV values:",
    (
        trade_features_table[
            "past_amount_cv"
        ] < 0
    ).sum(),
)

Historical sizing CV available: 31957
Invalid CV values: 0


In [232]:
display(
    trade_features_table.loc[
        trade_features_table[
            "past_amount_cv_available"
        ],
        [
            "past_amount_count",
            "past_amount_mean",
            "past_amount_std",
            "past_amount_cv",
        ],
    ].describe()
)

,past_amount_count,past_amount_mean,past_amount_std,past_amount_cv
count,31957.000000,31957.000000,31957.000000,31957.000000
mean,9.102794,0.161498,0.032660,0.219719
std,10.484199,0.118471,0.047677,0.265206
min,2.000000,0.010000,0.000000,0.000000
25%,3.000000,0.080000,0.000000,0.000000
50%,6.000000,0.118333,0.013093,0.134999
75%,11.000000,0.201786,0.047140,0.353553
max,151.000000,0.615000,0.295000,2.687775


In [233]:
display(
    trade_features_table[
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "open_date_time",
            "amount",
            "past_amount_count",
            "past_amount_mean",
            "past_amount_std",
            "past_amount_cv",
            "reverse_profit_per_lot",
        ]
    ].head(30)
)

,trade_row_id,account_id,campaign_id,open_date_time,amount,past_amount_count,past_amount_mean,past_amount_std,past_amount_cv,reverse_profit_per_lot
0,1,D#1589378,33,2026-02-24 12:01:08+00:00,0.01,0,NaN,NaN,NaN,-44.0
1,2,D#1589408,33,2026-02-24 02:46:35+00:00,0.43,0,NaN,NaN,NaN,395.0
2,3,D#1589408,33,2026-02-24 02:53:40+00:00,0.10,1,0.430000,NaN,NaN,-95.0
3,4,D#1589408,33,2026-02-24 02:57:37+00:00,0.30,2,0.265000,0.165000,0.622642,198.0
4,5,D#1589410,33,2026-02-25 00:13:04+00:00,0.10,0,NaN,NaN,NaN,-496.0
5,6,D#1589410,33,2026-02-25 00:23:39+00:00,0.10,1,0.100000,NaN,NaN,-464.0
6,7,D#1589419,33,2026-02-24 05:32:22+00:00,0.01,0,NaN,NaN,NaN,1968.0
7,8,D#1589428,33,2026-02-24 02:38:47+00:00,0.10,0,NaN,NaN,NaN,-172.0
8,9,D#1589428,33,2026-02-24 07:24:19+00:00,0.15,1,0.100000,NaN,NaN,-764.0
9,10,D#1589428,33,2026-02-24 08:38:11+00:00,0.15,2,0.125000,0.025000,0.200000,2187.0


In [234]:
sizing_consistency_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_amount_count"
        ].ge(3)
        & trade_features_table[
            "past_amount_cv"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_amount_count",
            "past_amount_cv",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [235]:
print(
    "Eligible trades:",
    len(
        sizing_consistency_test_data
    ),
)

print(
    "Eligible campaigns:",
    sizing_consistency_test_data[
        "campaign_id"
    ].nunique(),
)

Eligible trades: 26861
Eligible campaigns: 34


In [236]:
def learn_high_sizing_variability_threshold(
    train_data: pd.DataFrame,
    candidate_quantiles: tuple = (
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
    ),
    min_signal_trades: int = 20,
    min_control_trades: int = 20,
) -> dict | None:
    """Learns a threshold for unusually inconsistent historical sizing."""
    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = train_data[
            "past_amount_cv"
        ].quantile(
            quantile
        )

        signal_mask = (
            train_data[
                "past_amount_cv"
            ]
            >= threshold
        )

        control_mask = ~signal_mask

        signal_count = signal_mask.sum()
        control_count = control_mask.sum()

        if (
            signal_count < min_signal_trades
            or control_count < min_control_trades
        ):
            continue

        signal_mean = train_data.loc[
            signal_mask,
            "reverse_profit_per_lot",
        ].mean()

        control_mean = train_data.loc[
            control_mask,
            "reverse_profit_per_lot",
        ].mean()

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": signal_count,
                "training_control_count": control_count,
                "training_uplift": (
                    signal_mean
                    - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: result[
            "training_uplift"
        ],
    )

In [237]:
def walk_forward_high_sizing_variability(
    data: pd.DataFrame,
    minimum_training_campaigns: int = 10,
) -> pd.DataFrame:
    """Walk-forward tests high historical position-size variability."""
    campaign_order = (
        data["campaign_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    fold_results = []

    for test_index in range(
        minimum_training_campaigns,
        len(campaign_order),
    ):
        training_campaigns = (
            campaign_order[:test_index]
        )

        test_campaign = (
            campaign_order[test_index]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_high_sizing_variability_threshold(
                train_data
            )
        )

        if threshold_result is None:
            continue

        threshold = (
            threshold_result[
                "threshold"
            ]
        )

        signal_mask = (
            test_data[
                "past_amount_cv"
            ]
            >= threshold
        )

        signal_data = (
            test_data.loc[
                signal_mask
            ]
        )

        control_data = (
            test_data.loc[
                ~signal_mask
            ]
        )

        signal_mean = (
            signal_data[
                "reverse_profit_per_lot"
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                "reverse_profit_per_lot"
            ].mean()
            if not control_data.empty
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": len(
                    signal_data
                ),
                "test_control_trades": len(
                    control_data
                ),
                "test_uplift_vs_control": (
                    signal_mean
                    - control_mean
                    if (
                        not np.isnan(
                            signal_mean
                        )
                        and not np.isnan(
                            control_mean
                        )
                    )
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

In [238]:
sizing_consistency_walk_forward = (
    walk_forward_high_sizing_variability(
        sizing_consistency_test_data,
        minimum_training_campaigns=10,
    )
)

display(
    sizing_consistency_walk_forward
)

summarize_walk_forward(
    sizing_consistency_walk_forward,
    "High historical sizing variability",
)

,test_campaign,learned_quantile,learned_threshold,test_signal_trades,test_control_trades,test_uplift_vs_control
0,43,0.5,0.164526,363,493,26.613815
1,44,0.5,0.157459,471,464,34.148442
2,45,0.5,0.157459,293,425,88.302947
3,46,0.5,0.150585,367,387,-4.776595
4,47,0.5,0.149971,283,221,55.931882
5,48,0.5,0.152908,351,515,19.015606
6,49,0.5,0.145413,311,399,-6.619198
7,50,0.5,0.142857,307,279,-17.406130
8,51,0.5,0.144088,343,300,77.106610
9,52,0.5,0.144248,315,220,68.098198


High historical sizing variability
Mean OOS uplift: 25.99270315421761
Median OOS uplift: 16.273574606938702
Positive campaign share: 0.75
95% bootstrap CI: (np.float64(6.46800006778357), np.float64(49.846109461769686))
